# New Experiments Notebook
Priority order: (1) Periodic component forecasting on Weather, (2) Signal-statistics-matched Lorenz control, (3) Lambda1 estimation + forecast on Burgers, (4) Multi-seed subsampling, (5) Better period projection.


In [1]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.fft import fft, ifft, fftfreq
from scipy.linalg import svd
from scipy.stats import wilcoxon, linregress
from scipy.integrate import solve_ivp
from sklearn.metrics import pairwise_distances
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

N_WINDOWS   = 8
CONTEXT_LEN = 512
PRED_LEN    = 96
DATA_DIR    = './ts_data'  # adjust if needed

Device: cpu


In [2]:
import sys
sys.path.insert(0, './panda')

from panda.patchtst.pipeline import PatchTSTPipeline
from chronos import ChronosPipeline

panda_model = PatchTSTPipeline.from_pretrained(
    mode='predict',
    pretrain_path='GilpinLab/panda',
    device_map=device,
)

chronos_model = ChronosPipeline.from_pretrained(
    'amazon/chronos-t5-small',
    device_map=device,
    torch_dtype=torch.bfloat16,
)

print('Models loaded.')

Models loaded.


In [3]:
# -------------------------------------------------------
# Metrics
# -------------------------------------------------------
def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def mse(y_true, y_pred):
    return float(np.mean((y_true - y_pred)**2))

# -------------------------------------------------------
# Per-window normalisation
# -------------------------------------------------------
def instance_norm_window(x_CT):
    """x_CT: (C, T). Normalise per channel using this window only."""
    mu  = x_CT.mean(axis=1, keepdims=True)
    std = x_CT.std( axis=1, keepdims=True) + 1e-8
    return (x_CT - mu) / std, mu, std

def load_ts(path):
    """Raw (C, T) — no global normalisation."""
    df = pd.read_csv(path)
    df = df.select_dtypes(include=[np.number])
    return df.values.astype(np.float32).T  # (C, T)

# -------------------------------------------------------
# Inference
# -------------------------------------------------------
def panda_forecast(context_np, horizon):
    """context_np: (C, T) normalised. Returns (C, horizon)."""
    TRAIN_H   = 128
    remaining = horizon
    ctx       = context_np.copy()
    preds     = []
    while remaining > 0:
        h         = min(TRAIN_H, remaining)
        context_t = torch.tensor(ctx.T, dtype=torch.float32)
        with torch.no_grad():
            pred = panda_model.predict(
                context_t, h,
                limit_prediction_length=False,
                sliding_context=True,
            )
        p = pred.squeeze().cpu().numpy()
        if p.ndim == 1:
            p = p[:, None]
        if p.shape[0] != context_np.shape[0]:
            p = p.T
        preds.append(p[:, :h])
        ctx       = np.concatenate([ctx[:, h:], p[:, :h]], axis=1)
        remaining -= h
    return np.concatenate(preds, axis=1)  # (C, horizon)

def chronos_forecast(context_np, horizon):
    """Batched — all channels in one call."""
    ctx = torch.tensor(context_np, dtype=torch.float32)
    with torch.no_grad():
        out = chronos_model.predict(
            ctx, prediction_length=horizon, num_samples=1
        )
    return out[:, 0, :].cpu().numpy()  # (C, horizon)

# -------------------------------------------------------
# Core evaluator
# -------------------------------------------------------
def evaluate(data_CT, horizon, n_windows=N_WINDOWS, label='',
             fn_a=None, fn_b=None,
             name_a='panda', name_b='chronos'):
    """
    data_CT: (C, T) RAW. Normalises each window independently.
    fn_a, fn_b: (context_normed: (C,T), horizon) -> (C, H)
    """
    if fn_a is None: fn_a = panda_forecast
    if fn_b is None: fn_b = chronos_forecast

    C, T      = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    if max_start <= 0:
        print(f'  [SKIP] {label}: T={T} too short')
        return None

    starts = np.linspace(0, max_start, n_windows, dtype=int)
    mae_a, mae_b = [], []

    for s in starts:
        ctx_raw           = data_CT[:, s : s + CONTEXT_LEN]
        tgt_raw           = data_CT[:, s + CONTEXT_LEN : s + CONTEXT_LEN + horizon]
        ctx_norm, mu, std = instance_norm_window(ctx_raw)
        tgt_norm          = (tgt_raw - mu) / std
        mae_a.append(mae(tgt_norm, fn_a(ctx_norm, horizon)))
        mae_b.append(mae(tgt_norm, fn_b(ctx_norm, horizon)))

    diff = np.array(mae_b) - np.array(mae_a)
    try:
        _, pval = wilcoxon(diff, alternative='greater') \
            if np.any(diff != 0) else (0, 1.0)
    except Exception:
        pval = np.nan

    adv = np.median(mae_b) - np.median(mae_a)
    sig = ' *' if pval < 0.05 else (' ~' if pval < 0.10 else '')
    iqr_a = np.percentile(mae_a,75) - np.percentile(mae_a,25)
    iqr_b = np.percentile(mae_b,75) - np.percentile(mae_b,25)

    result = {
        'label'         : label,
        'horizon'       : horizon,
        'name_a'        : name_a,
        'name_b'        : name_b,
        f'{name_a}_mae' : np.median(mae_a),
        f'{name_a}_iqr' : iqr_a,
        f'{name_b}_mae' : np.median(mae_b),
        f'{name_b}_iqr' : iqr_b,
        'advantage_mae' : adv,
        'wilcoxon_p'    : pval,
    }
    print(
        f'  {label:50s}  H={horizon:4d}  '
        f'{name_a}={np.median(mae_a):.4f}[±{iqr_a:.4f}]  '
        f'{name_b}={np.median(mae_b):.4f}[±{iqr_b:.4f}]  '
        f'Adv={adv:+.4f}  p={pval:.3f}{sig}'
    )
    return result

print('Helpers defined.')

Helpers defined.


## Priority 1: Direct Periodic Component Forecasting on Weather
**Question:** Is Panda's advantage on Weather due to better periodic handling, or does it come from the residual/stochastic component?
**Method:** Extract deterministic (periodic) component via FFT from context window. Run both models on just that component. Compare advantage on full vs periodic-only signal.


In [4]:
def extract_periodic_component(ctx_1d, n_harmonics=5):
    """Keep top-n_harmonics frequency components (excluding DC) from context."""
    N   = len(ctx_1d)
    X   = np.fft.rfft(ctx_1d)
    mag = np.abs(X.copy())
    mag[0] = 0  # exclude DC
    top_idx = np.argsort(mag)[-n_harmonics:]
    X_filt        = np.zeros_like(X)
    X_filt[0]     = X[0]  # keep mean
    X_filt[top_idx] = X[top_idx]
    return np.fft.irfft(X_filt, n=N).astype(np.float32)

def project_periodic_future(ctx_1d, pred_len, n_harmonics=5):
    """Extend periodic component into future using fitted sinusoids from context only."""
    N     = len(ctx_1d)
    X     = np.fft.rfft(ctx_1d)
    freqs = np.fft.rfftfreq(N)
    mag   = np.abs(X.copy())
    mag[0] = 0
    top_idx   = np.argsort(mag)[-n_harmonics:]
    t_future  = np.arange(N, N + pred_len)
    projection = np.real(X[0]) / N  # DC
    for idx in top_idx:
        amp   = np.abs(X[idx]) / N * 2
        phase = np.angle(X[idx])
        projection = projection + amp * np.cos(2 * np.pi * freqs[idx] * t_future + phase)
    return projection.astype(np.float32)

def build_periodic_windows_CT(data_CT, n_windows, horizon, n_harmonics=5):
    """
    Returns two (C, T_window) arrays per window: periodic context and periodic target.
    Same window starts as evaluate() uses (linspace).
    """
    C, T      = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    starts    = np.linspace(0, max_start, n_windows, dtype=int)
    windows   = []
    for s in starts:
        ctx_raw = data_CT[:, s : s + CONTEXT_LEN]
        ctx_per = np.zeros_like(ctx_raw)
        tgt_per = np.zeros((C, horizon), dtype=np.float32)
        for c in range(C):
            ctx_per[c] = extract_periodic_component(ctx_raw[c], n_harmonics)
            tgt_per[c] = project_periodic_future(ctx_raw[c], horizon, n_harmonics)
        windows.append((ctx_per, tgt_per))
    return windows

def evaluate_periodic(data_CT, horizon, n_windows=N_WINDOWS,
                       label='', n_harmonics=5):
    """Run both models on periodic-component-only signal."""
    C, T      = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    if max_start <= 0:
        print(f'  [SKIP] {label}: too short')
        return None

    windows  = build_periodic_windows_CT(data_CT, n_windows, horizon, n_harmonics)
    mae_p, mae_c = [], []

    for ctx_per, tgt_per in windows:
        ctx_norm, mu, std = instance_norm_window(ctx_per)
        tgt_norm          = (tgt_per - mu) / std
        mae_p.append(mae(tgt_norm, panda_forecast(ctx_norm, horizon)))
        mae_c.append(mae(tgt_norm, chronos_forecast(ctx_norm, horizon)))

    diff = np.array(mae_c) - np.array(mae_p)
    try:
        _, pval = wilcoxon(diff, alternative='greater') \
            if np.any(diff != 0) else (0, 1.0)
    except Exception:
        pval = np.nan

    adv   = np.median(mae_c) - np.median(mae_p)
    iqr_p = np.percentile(mae_p,75) - np.percentile(mae_p,25)
    iqr_c = np.percentile(mae_c,75) - np.percentile(mae_c,25)
    sig   = ' *' if pval < 0.05 else (' ~' if pval < 0.10 else '')
    print(
        f'  {label:50s}  H={horizon:4d}  '
        f'panda={np.median(mae_p):.4f}[±{iqr_p:.4f}]  '
        f'chronos={np.median(mae_c):.4f}[±{iqr_c:.4f}]  '
        f'Adv={adv:+.4f}  p={pval:.3f}{sig}'
    )
    return {
        'label': label, 'horizon': horizon,
        'panda_mae': np.median(mae_p), 'panda_iqr': iqr_p,
        'chronos_mae': np.median(mae_c), 'chronos_iqr': iqr_c,
        'advantage_mae': adv, 'wilcoxon_p': pval,
    }

print('Priority 1 helpers defined.')

Priority 1 helpers defined.


In [5]:
print('Priority 1: Periodic Component Forecasting — Weather')
print('-' * 70)

data_weather = load_ts(f'{DATA_DIR}/weather.csv')
print(f'Weather shape: {data_weather.shape}')

p1_results = []

for h in [96, 336]:
    print(f'\n  H={h}:')

    # Full signal
    r_full = evaluate(data_weather, h, n_windows=N_WINDOWS,
                      label=f'Weather_full_H{h}')
    if r_full:
        r_full['condition'] = 'full'
        r_full['dataset']   = 'Weather'
        p1_results.append(r_full)

    # Periodic component only
    r_per = evaluate_periodic(data_weather, h, n_windows=N_WINDOWS,
                               label=f'Weather_periodic_H{h}', n_harmonics=5)
    if r_per:
        r_per['condition'] = 'periodic_only'
        r_per['dataset']   = 'Weather'
        p1_results.append(r_per)

df_p1 = pd.DataFrame(p1_results)
df_p1.to_csv('p1_periodic_results.csv', index=False)
print('\nSaved p1_periodic_results.csv')

print('\n=== Priority 1 Summary ===')
for h in [96, 336]:
    full = df_p1[(df_p1.condition=='full') & (df_p1.horizon==h)]
    per  = df_p1[(df_p1.condition=='periodic_only') & (df_p1.horizon==h)]
    if len(full) and len(per):
        adv_full = float(full.advantage_mae)
        adv_per  = float(per.advantage_mae)
        ratio    = adv_per / (adv_full + 1e-8)
        print(f'  H={h}: Adv(full)={adv_full:+.4f}  Adv(periodic)={adv_per:+.4f}  ratio={ratio:.2f}')
        if ratio < 0.1:
            obs = 'Advantage collapses on periodic component -> Panda advantage is from residual dynamics.'
        elif ratio > 0.5 and adv_per > 0:
            obs = 'Advantage persists on periodic component -> Panda handles periodicity better than Chronos.'
        else:
            obs = 'Partial effect. Both components contribute.'
        print(f'  Observation: {obs}')

Priority 1: Periodic Component Forecasting — Weather
----------------------------------------------------------------------
Weather shape: (21, 52696)

  H=96:


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Weather_full_H96                                    H=  96  panda=0.6128[±0.2428]  chronos=0.8021[±0.2334]  Adv=+0.1893  p=0.004 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Weather_periodic_H96                                H=  96  panda=0.1757[±0.0582]  chronos=0.6978[±0.1260]  Adv=+0.5220  p=0.004 *

  H=336:


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Weather_full_H336                                   H= 336  panda=0.8754[±0.2712]  chronos=0.9786[±0.3031]  Adv=+0.1031  p=0.020 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Weather_periodic_H336                               H= 336  panda=0.4930[±0.1401]  chronos=0.9944[±0.1062]  Adv=+0.5014  p=0.004 *

Saved p1_periodic_results.csv

=== Priority 1 Summary ===
  H=96: Adv(full)=+0.1893  Adv(periodic)=+0.5220  ratio=2.76
  Observation: Advantage persists on periodic component -> Panda handles periodicity better than Chronos.
  H=336: Adv(full)=+0.1031  Adv(periodic)=+0.5014  ratio=4.86
  Observation: Advantage persists on periodic component -> Panda handles periodicity better than Chronos.


## Priority 2: Signal-Statistics-Matched Lorenz Control
**Question:** Does Panda win on Lorenz because of chaotic dynamics, or just because of the signal statistics (power spectrum)?
**Method:** Phase-randomization surrogate — matched power spectrum, randomised phases — destroys dynamical structure while preserving statistics.


In [5]:
def lorenz_rhs(t, state, sigma=10.0, rho=28.0, beta=8.0/3.0):
    x, y, z = state
    return [sigma*(y-x), x*(rho-z)-y, x*y-beta*z]

def simulate_lorenz(n_steps=6000, dt=0.01, rho=28.0, seed=SEED):
    rng    = np.random.default_rng(seed)
    ic     = rng.standard_normal(3)
    t_span = (0, n_steps * dt)
    t_eval = np.linspace(*t_span, n_steps)
    sol    = solve_ivp(lorenz_rhs, t_span, ic, t_eval=t_eval,
                       args=(10.0, rho, 8.0/3.0),
                       method='RK45', rtol=1e-9, atol=1e-9)
    return sol.y[0].astype(np.float32)  # x component

def phase_shuffle_surrogate(series, seed=SEED):
    """
    Phase-randomization surrogate.
    Matches power spectrum exactly; destroys temporal ordering.
    """
    rng     = np.random.default_rng(seed)
    N       = len(series)
    X       = np.fft.rfft(series)
    phases  = rng.uniform(0, 2*np.pi, len(X))
    X_shuf  = np.abs(X) * np.exp(1j * phases)
    surr    = np.fft.irfft(X_shuf, n=N).astype(np.float32)
    # Rescale to exactly match original mean/std
    surr    = (surr - surr.mean()) / (surr.std() + 1e-8)
    surr    = surr * series.std() + series.mean()
    return surr

print('Priority 2 helpers defined.')

Priority 2 helpers defined.


In [7]:
print('Priority 2: Signal-Statistics-Matched Lorenz Control')
print('-' * 70)

lorenz_series   = simulate_lorenz(n_steps=6000, rho=28.0)
surrogate_series = phase_shuffle_surrogate(lorenz_series)

print(f'Lorenz   mean={lorenz_series.mean():.3f}  std={lorenz_series.std():.3f}')
print(f'Surrogate mean={surrogate_series.mean():.3f}  std={surrogate_series.std():.3f}')

# Wrap as (1, T) for evaluate()
lorenz_CT   = lorenz_series[None, :]    # (1, T)
surrogate_CT = surrogate_series[None, :]  # (1, T)

p2_results = []

print('\n  Lorenz rho=28 (chaotic):')
r_lor = evaluate(lorenz_CT, PRED_LEN, n_windows=N_WINDOWS,
                  label='Lorenz_rho28_chaotic')
if r_lor:
    r_lor['condition'] = 'chaotic'
    p2_results.append(r_lor)

print('\n  Phase-shuffled surrogate (stats-matched):')
r_sur = evaluate(surrogate_CT, PRED_LEN, n_windows=N_WINDOWS,
                  label='Lorenz_surrogate_statsmatched')
if r_sur:
    r_sur['condition'] = 'surrogate'
    p2_results.append(r_sur)

df_p2 = pd.DataFrame(p2_results)
df_p2.to_csv('p2_lorenz_surrogate_results.csv', index=False)
print('\nSaved p2_lorenz_surrogate_results.csv')

print('\n=== Priority 2 Summary ===')
adv_lor = float(df_p2[df_p2.condition=='chaotic'].advantage_mae)
adv_sur = float(df_p2[df_p2.condition=='surrogate'].advantage_mae)
ratio   = adv_sur / (adv_lor + 1e-8)
print(f'  Adv(Lorenz chaotic):    {adv_lor:+.4f}')
print(f'  Adv(surrogate):         {adv_sur:+.4f}')
print(f'  Surrogate/Lorenz ratio: {ratio:.2f}')
if ratio > 0.5:
    obs = 'Advantage persists on surrogate. Signal statistics (not chaotic dynamics) drive Panda advantage.'
elif ratio < 0.1:
    obs = 'Advantage collapses on surrogate. Advantage is specific to chaotic dynamical structure.'
else:
    obs = 'Partial collapse. Both signal statistics and dynamical structure contribute.'
print(f'  Observation: {obs}')

Priority 2: Signal-Statistics-Matched Lorenz Control
----------------------------------------------------------------------


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


Lorenz   mean=2.451  std=7.621
Surrogate mean=2.451  std=7.621

  Lorenz rho=28 (chaotic):


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Lorenz_rho28_chaotic                                H=  96  panda=0.0558[±0.0533]  chronos=0.4393[±1.3623]  Adv=+0.3835  p=0.004 *

  Phase-shuffled surrogate (stats-matched):


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


  Lorenz_surrogate_statsmatched                       H=  96  panda=0.7279[±0.2445]  chronos=0.8995[±0.6550]  Adv=+0.1715  p=0.320

Saved p2_lorenz_surrogate_results.csv

=== Priority 2 Summary ===
  Adv(Lorenz chaotic):    +0.3835
  Adv(surrogate):         +0.1715
  Surrogate/Lorenz ratio: 0.45
  Observation: Partial collapse. Both signal statistics and dynamical structure contribute.


## Priority 3: Lambda1 Estimation + Forecast on Burgers
**Question:** Does Panda win on Burgers at non-chaotic viscosity (lambda1 < 0)? If yes, chaos-specific hypothesis is falsified for PDEs.
**Method:** Simulate Burgers at multiple nu values, extract first PCA component, estimate lambda1 via corrected Rosenstein, then run Panda vs Chronos.


In [6]:
def simulate_burgers_stable(T=1000, N_x=128, nu=0.005, seed=SEED):
    """
    1D viscous Burgers via spectral method with adaptive dt.
    Stable at any nu including nu=1.0, 2.0.
    """
    rng = np.random.default_rng(seed)
    dx  = 2 * np.pi / N_x

    dt_diff   = 0.4 * dx**2 / (2 * nu + 1e-10)
    dt_adv    = 0.4 * dx
    dt        = min(dt_diff, dt_adv, 0.05)
    dt_record = 0.01
    n_sub     = max(1, int(np.ceil(dt_record / dt)))
    dt_act    = dt_record / n_sub

    k       = fftfreq(N_x, d=1.0/N_x).astype(complex)
    dealias = np.abs(k) <= N_x // 3
    L_op    = -nu * k**2

    u0_hat = np.zeros(N_x, dtype=complex)
    for m in range(1, 6):
        amp = rng.standard_normal() + 1j * rng.standard_normal()
        u0_hat[m]       += amp
        u0_hat[N_x - m] += np.conj(amp)
    u0_hat *= dealias

    def rhs_hat(u_hat):
        u_phys = np.real(ifft(u_hat))
        nonlin = fft(0.5 * u_phys**2) * dealias
        return L_op * u_hat - 1j * k * nonlin

    U     = np.zeros((T, N_x), dtype=np.float32)
    u_hat = u0_hat.copy()
    for t in range(T):
        U[t] = np.real(ifft(u_hat)).astype(np.float32)
        for _ in range(n_sub):
            k1    = rhs_hat(u_hat)
            k2    = rhs_hat(u_hat + 0.5*dt_act*k1)
            k3    = rhs_hat(u_hat + 0.5*dt_act*k2)
            k4    = rhs_hat(u_hat +     dt_act*k3)
            u_hat = u_hat + (dt_act/6.0)*(k1+2*k2+2*k3+k4)
            u_hat *= dealias
            if not np.isfinite(u_hat).all():
                print(f'    Diverged at t={t}')
                return U[:t]
    return U

def pca_reduction(U, n_components):
    U_c  = U - U.mean(axis=0, keepdims=True)
    n_c  = min(n_components, min(U_c.shape)-1)
    _, _, Vt = svd(U_c, full_matrices=False)
    return (U_c @ Vt[:n_c].T).astype(np.float32)  # (T, n_c)

def rosenstein_lambda1(series, m=3, tau=1, max_iter=50):
    """
    Corrected Rosenstein estimator for largest Lyapunov exponent.
    m: embedding dim, tau: delay, max_iter: divergence steps.
    Returns lambda1 in log-units/step (positive = chaotic).
    """
    N      = len(series)
    n_emb  = N - (m-1)*tau
    if n_emb < max_iter + 20:
        return np.nan

    embedded = np.array([series[i : i+(m-1)*tau+1 : tau] for i in range(n_emb)])
    w        = max(int(N * 0.02), 10)
    divergences = []

    for i in range(n_emb - max_iter):
        dists = np.linalg.norm(embedded - embedded[i], axis=1)
        dists[max(0,i-w):min(n_emb,i+w)] = np.inf
        nn = np.argmin(dists)
        if dists[nn] == np.inf:
            continue
        div = []
        for step in range(max_iter):
            if i+step >= n_emb or nn+step >= n_emb:
                break
            d = np.linalg.norm(embedded[i+step] - embedded[nn+step])
            div.append(np.log(d + 1e-12))
        if div:
            divergences.append(div)

    if not divergences:
        return np.nan

    min_len    = min(len(d) for d in divergences)
    if min_len < 3:
        return np.nan

    div_matrix = np.array([d[:min_len] for d in divergences])
    avg_div    = div_matrix.mean(axis=0)          # shape (min_len,)
    linear_end = max(3, min_len // 2)
    linear_end = min(linear_end, len(avg_div))    # <-- the actual fix

    t = np.arange(linear_end)
    y = avg_div[:linear_end]
    assert len(t) == len(y), f"Shape mismatch: t={len(t)}, y={len(y)}"

    slope, _, _, _, _ = linregress(t, y)
    return float(slope)

print("rosenstein_lambda1 redefined.")
# Test solver
print('Testing Burgers solver:')
for nu_test in [2.0, 1.0, 0.1, 0.01, 0.005]:
    U      = simulate_burgers_stable(T=50, nu=nu_test)
    status = 'OK' if len(U) == 50 else f'FAILED at step {len(U)}'
    print(f'  nu={nu_test:.3f}: range=[{U.min():.3f},{U.max():.3f}]  {status}')

rosenstein_lambda1 redefined.
Testing Burgers solver:
  nu=2.000: range=[-0.060,0.063]  OK
  nu=1.000: range=[-0.060,0.063]  OK
  nu=0.100: range=[-0.060,0.063]  OK
  nu=0.010: range=[-0.060,0.063]  OK
  nu=0.005: range=[-0.060,0.063]  OK


In [11]:
print('Priority 3: Lambda1 Estimation on Burgers PCA Components')
print('-' * 70)

# nu >= 0.5: non-chaotic; nu < 0.1: transitioning to chaotic
nu_values    = [2.0, 1.0, 0.5, 0.1, 0.05, 0.01, 0.005]
N_COMP       = 16
T_SIM        = 1500  # keep reasonable for CPU
lambda1_map  = {}
p3_results   = []

for nu in nu_values:
    print(f'\n  nu={nu}:')
    U = simulate_burgers_stable(T=T_SIM, N_x=128, nu=nu)
    if len(U) < CONTEXT_LEN + PRED_LEN + 10:
        print(f'    Too short ({len(U)} steps), skipping')
        continue

    # PCA: (T, N_COMP)
    pca_series = pca_reduction(U, N_COMP)
    pc1        = pca_series[:, 0]  # first component (T,)

    # Explained variance
    U_c     = U - U.mean(axis=0, keepdims=True)
    _, sv, _ = svd(U_c, full_matrices=False)
    ev_ratio = sv[0]**2 / (sv**2).sum()
    print(f'    PC1 explained variance: {ev_ratio:.3f}')

    # Lambda1 on PC1
    lam = rosenstein_lambda1(pc1, m=3, tau=1, max_iter=50)
    lambda1_map[nu] = lam
    sign = 'positive (chaotic)' if (lam is not np.nan and lam > 0) else 'negative/zero (non-chaotic)'
    print(f'    Lambda1: {lam:.4f}  ({sign})')

    # Forecast on PCA components (C, T) format
    data_CT = pca_series.T  # (N_COMP, T)
    res = evaluate(data_CT, PRED_LEN, n_windows=N_WINDOWS,
                   label=f'Burgers_PCA_nu={nu:.3f}')
    if res:
        res['nu']      = nu
        res['lambda1'] = lam
        p3_results.append(res)

df_p3 = pd.DataFrame(p3_results)
df_p3.to_csv('p3_burgers_lambda1_results.csv', index=False)
print('\nSaved p3_burgers_lambda1_results.csv')

print('\n=== Priority 3 Summary ===')
print(f'{"nu":>6} | {"lambda1":>8} | {"chaotic":>7} | {"advantage":>10} | {"p_val":>7}')
print('-' * 50)
for _, row in df_p3.iterrows():
    ch = 'yes' if row.lambda1 > 0 else 'no'
    print(f'{row.nu:>6} | {row.lambda1:>8.3f} | {ch:>7} | {row.advantage_mae:>10.4f} | {row.wilcoxon_p:>7.4f}')

# Key check: does Panda win at nu where lambda1 < 0?
non_chaotic_rows = df_p3[df_p3.lambda1 < 0]
if len(non_chaotic_rows):
    if (non_chaotic_rows.advantage_mae > 0.05).any():
        print('\nObservation: Panda wins at non-chaotic viscosity (lambda1 < 0). Chaos-specific hypothesis falsified for PDEs.')
    else:
        print('\nObservation: Panda does not win at non-chaotic viscosity. Consistent with chaos-specific advantage.')
else:
    print('\nObservation: All tested nu values appear chaotic. Reduce to higher nu values to find non-chaotic regime.')

Priority 3: Lambda1 Estimation on Burgers PCA Components
----------------------------------------------------------------------

  nu=2.0:
    PC1 explained variance: 0.819
    Lambda1: nan  (negative/zero (non-chaotic))


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Burgers_PCA_nu=2.000                                H=  96  panda=0.0092[±0.0042]  chronos=0.0077[±0.0231]  Adv=-0.0015  p=0.191

  nu=1.0:
    PC1 explained variance: 0.821
    Lambda1: -0.0114  (negative/zero (non-chaotic))


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Burgers_PCA_nu=1.000                                H=  96  panda=0.0164[±0.0043]  chronos=0.0210[±0.0207]  Adv=+0.0045  p=0.004 *

  nu=0.5:
    PC1 explained variance: 0.824
    Lambda1: -0.0067  (negative/zero (non-chaotic))


...

  Burgers_PCA_nu=0.050                                H=  96  panda=0.0593[±0.0189]  chronos=0.1780[±0.0631]  Adv=+0.1187  p=0.004 *

  nu=0.01:
    PC1 explained variance: 0.973
    Lambda1: -0.0010  (negative/zero (non-chaotic))


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Burgers_PCA_nu=0.010                                H=  96  panda=0.0755[±0.0213]  chronos=0.2266[±0.1198]  Adv=+0.1510  p=0.004 *

  nu=0.005:
    PC1 explained variance: 0.945
    Lambda1: -0.0007  (negative/zero (non-chaotic))


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Burgers_PCA_nu=0.005                                H=  96  panda=0.0782[±0.0262]  chronos=0.1786[±0.0472]  Adv=+0.1003  p=0.004 *

Saved p3_burgers_lambda1_results.csv

=== Priority 3 Summary ===
    nu |  lambda1 | chaotic |  advantage |   p_val
--------------------------------------------------
   2.0 |      nan |      no |    -0.0015 |  0.1914
   1.0 |   -0.011 |      no |     0.0045 |  0.0039
   0.5 |   -0.007 |      no |     0.0156 |  0.0117
   0.1 |   -0.004 |      no |     0.1010 |  0.0039
  0.05 |   -0.003 |      no |     0.1187 |  0.0039
  0.01 |   -0.001 |      no |     0.1510 |  0.0039
 0.005 |   -0.001 |      no |     0.1003 |  0.0039

Observation: Panda wins at non-chaotic viscosity (lambda1 < 0). Chaos-specific hypothesis falsified for PDEs.


## Priority 4: Multiple Seeds for Subsampling
**Question:** Is the single-seed subsampling result (Diversity gives larger relative advantage) reliable, or is it high-variance?
**Method:** Run Diversity and Stratified_Uniform at 10 seeds. Measure variance in Panda absolute MAE and advantage across seeds.


In [7]:
def variance_stratified_subsample(U, n_points, pct=10):
    """Uniform spacing excluding bottom pct% variance locations."""
    variances = U.var(axis=0)
    threshold = np.percentile(variances, pct)
    valid     = np.where(variances >= threshold)[0]
    if len(valid) < n_points:
        valid = np.arange(U.shape[1])
    selected  = valid[np.linspace(0, len(valid)-1, n_points, dtype=int)]
    return U[:, selected].astype(np.float32), selected

def compute_dynamical_features(U):
    from scipy.signal import periodogram
    T_u, N_x = U.shape
    features  = []
    for x in range(N_x):
        ts = U[:, x].astype(float)
        freqs, power = periodogram(ts)
        p_norm = power / (power.sum() + 1e-10)
        p_norm = p_norm[p_norm > 1e-10]
        features.append([
            float(np.std(ts)),
            float(np.mean(np.abs(ts))),
            float(np.percentile(np.abs(ts), 90)),
            float(-np.sum(p_norm * np.log(p_norm))),
            float(freqs[np.argmax(power[1:])+1]),
        ])
    F = np.array(features)
    return (F - F.min(0)) / (F.max(0) - F.min(0) + 1e-8)

def farthest_point_sampling(features, n_points, seed=SEED):
    rng      = np.random.default_rng(seed)
    N_x      = features.shape[0]
    selected = [int(rng.integers(0, N_x))]
    dists    = np.full(N_x, np.inf)
    for _ in range(n_points - 1):
        last        = selected[-1]
        d           = np.linalg.norm(features - features[last], axis=1)
        dists       = np.minimum(dists, d)
        dists_copy  = dists.copy()
        dists_copy[selected] = -np.inf
        selected.append(int(np.argmax(dists_copy)))
    return sorted(selected)

def diversity_subsample(U, n_points, seed=SEED):
    features = compute_dynamical_features(U)
    indices  = farthest_point_sampling(features, n_points, seed=seed)
    return U[:, indices].astype(np.float32), indices

print('Priority 4 helpers defined.')

Priority 4 helpers defined.


In [13]:
print('Priority 4: Multi-Seed Subsampling Variance')
print('-' * 70)

N_CHANNELS = 16
SEEDS_LIST = list(range(10))
NU_SUB     = 0.05  # chaotic regime, same as fixed_experiments

print(f'Simulating Burgers nu={NU_SUB} T={T_SIM}...')
U_sub = simulate_burgers_stable(T=T_SIM, N_x=128, nu=NU_SUB)
print(f'Shape: {U_sub.shape}')

p4_records = []

for seed in SEEDS_LIST:
    print(f'\n  Seed {seed}:')

    # Diversity
    U_div, _   = diversity_subsample(U_sub, N_CHANNELS, seed=seed)
    data_div   = U_div.T  # (N_CHANNELS, T)
    r_div      = evaluate(data_div, PRED_LEN, n_windows=N_WINDOWS,
                           label=f'Diversity_seed{seed}')
    if r_div:
        r_div['method'] = 'diversity'
        r_div['seed']   = seed
        p4_records.append(r_div)

    # Stratified Uniform
    U_strat, _ = variance_stratified_subsample(U_sub, N_CHANNELS)
    data_strat = U_strat.T
    r_strat    = evaluate(data_strat, PRED_LEN, n_windows=N_WINDOWS,
                           label=f'Stratified_seed{seed}')
    if r_strat:
        r_strat['method'] = 'stratified_uniform'
        r_strat['seed']   = seed
        p4_records.append(r_strat)

df_p4 = pd.DataFrame(p4_records)
df_p4.to_csv('p4_subsampling_seeds_results.csv', index=False)
print('\nSaved p4_subsampling_seeds_results.csv')

print('\n=== Priority 4 Summary ===')
for method in ['diversity', 'stratified_uniform']:
    sub    = df_p4[df_p4.method == method]
    p_mae  = sub['panda_mae'].values
    adv    = sub['advantage_mae'].values
    cv     = p_mae.std() / (p_mae.mean() + 1e-8)
    print(f'\n  {method}:')
    print(f'    Panda MAE: mean={p_mae.mean():.4f}  std={p_mae.std():.4f}  CV={cv:.3f}')
    print(f'    Advantage: mean={adv.mean():.4f}  std={adv.std():.4f}')

div_cv = df_p4[df_p4.method=='diversity']['panda_mae'].std() / \
         df_p4[df_p4.method=='diversity']['panda_mae'].mean()
if div_cv < 0.05:
    obs = 'Panda MAE CV < 5%. Absolute performance stable across seeds. Subsampling method unlikely to affect Panda forecasting.'
else:
    obs = f'Panda MAE CV = {div_cv:.3f} (>5%). Seed variance substantial; single-seed subsampling conclusions unreliable.'
print(f'\n  Observation: {obs}')

Priority 4: Multi-Seed Subsampling Variance
----------------------------------------------------------------------
Simulating Burgers nu=0.05 T=1500...
Shape: (1500, 128)

  Seed 0:


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Diversity_seed0                                     H=  96  panda=0.0261[±0.0064]  chronos=0.0938[±0.0761]  Adv=+0.0677  p=0.004 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Stratified_seed0                                    H=  96  panda=0.0281[±0.0048]  chronos=0.0706[±0.0435]  Adv=+0.0424  p=0.004 *

  Seed 1:


...

  Diversity_seed6                                     H=  96  panda=0.0269[±0.0058]  chronos=0.0986[±0.0754]  Adv=+0.0716  p=0.004 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Stratified_seed6                                    H=  96  panda=0.0281[±0.0048]  chronos=0.0713[±0.0703]  Adv=+0.0432  p=0.008 *

  Seed 7:


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


KeyboardInterrupt: 

In [14]:
print(len(p4_records))  # should be 14 if seeds 0-6 complete

14


In [15]:
for seed in SEEDS_LIST[7:]:
    print(f'\n  Seed {seed}:')

    U_div, _   = diversity_subsample(U_sub, N_CHANNELS, seed=seed)
    data_div   = U_div.T
    r_div      = evaluate(data_div, PRED_LEN, n_windows=N_WINDOWS,
                           label=f'Diversity_seed{seed}')
    if r_div:
        r_div['method'] = 'diversity'
        r_div['seed']   = seed
        p4_records.append(r_div)

    U_strat, _ = variance_stratified_subsample(U_sub, N_CHANNELS)
    data_strat = U_strat.T
    r_strat    = evaluate(data_strat, PRED_LEN, n_windows=N_WINDOWS,
                           label=f'Stratified_seed{seed}')
    if r_strat:
        r_strat['method'] = 'stratified_uniform'
        r_strat['seed']   = seed
        p4_records.append(r_strat)

df_p4 = pd.DataFrame(p4_records)
df_p4.to_csv('p4_subsampling_seeds_results.csv', index=False)
print('\nSaved p4_subsampling_seeds_results.csv')

print('\n=== Priority 4 Summary ===')
for method in ['diversity', 'stratified_uniform']:
    sub    = df_p4[df_p4.method == method]
    p_mae  = sub['panda_mae'].values
    adv    = sub['advantage_mae'].values
    cv     = p_mae.std() / (p_mae.mean() + 1e-8)
    print(f'\n  {method}:')
    print(f'    Panda MAE: mean={p_mae.mean():.4f}  std={p_mae.std():.4f}  CV={cv:.3f}')
    print(f'    Advantage: mean={adv.mean():.4f}  std={adv.std():.4f}')

div_cv = df_p4[df_p4.method=='diversity']['panda_mae'].std() / \
         df_p4[df_p4.method=='diversity']['panda_mae'].mean()
if div_cv < 0.05:
    obs = 'Panda MAE CV < 5%. Absolute performance stable across seeds.'
else:
    obs = f'Panda MAE CV = {div_cv:.3f} (>5%). Seed variance substantial.'
print(f'\n  Observation: {obs}')


  Seed 7:


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Diversity_seed7                                     H=  96  panda=0.0304[±0.0074]  chronos=0.1450[±0.2452]  Adv=+0.1146  p=0.004 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Stratified_seed7                                    H=  96  panda=0.0281[±0.0048]  chronos=0.1010[±0.1586]  Adv=+0.0729  p=0.004 *

  Seed 8:


...

  Stratified_seed8                                    H=  96  panda=0.0281[±0.0048]  chronos=0.0809[±0.1233]  Adv=+0.0528  p=0.004 *

  Seed 9:


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Diversity_seed9                                     H=  96  panda=0.0293[±0.0063]  chronos=0.1090[±0.0537]  Adv=+0.0797  p=0.004 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Stratified_seed9                                    H=  96  panda=0.0281[±0.0048]  chronos=0.0606[±0.1067]  Adv=+0.0324  p=0.004 *

Saved p4_subsampling_seeds_results.csv

=== Priority 4 Summary ===

  diversity:
    Panda MAE: mean=0.0278  std=0.0014  CV=0.052
    Advantage: mean=0.0772  std=0.0281

  stratified_uniform:
    Panda MAE: mean=0.0281  std=0.0000  CV=0.000
    Advantage: mean=0.0564  std=0.0213

  Observation: Panda MAE CV = 0.054 (>5%). Seed variance substantial.


## Priority 5: Better Period Projection in Decomposition
**Question:** Does Panda's advantage survive when we use a better projection of the deterministic component (actual seasonal pattern from context, not naive repeat)?
**Method:** Instead of repeating the last period, use the average seasonal pattern from all complete periods in the context window. No oracle leakage — no future residuals used.


In [8]:
def extract_dominant_period(ctx_1d, max_period=None):
    """Estimate dominant period via FFT peak."""
    N          = len(ctx_1d)
    if max_period is None:
        max_period = N // 2
    X          = np.fft.rfft(ctx_1d)
    fft_mag    = np.abs(X)
    fft_mag[0] = 0
    freqs      = np.fft.rfftfreq(N)
    valid      = (freqs > 0) & (1.0/(freqs+1e-12) <= max_period)
    if not valid.any():
        return N // 4
    peak_freq  = freqs[valid][np.argmax(fft_mag[valid])]
    return max(2, int(round(1.0 / peak_freq)))

def project_seasonal_improved(ctx_1d, pred_len):
    """
    Average seasonal pattern over all complete periods in context.
    Phase-aligns projection to continue smoothly from ctx end.
    No oracle leakage: uses only ctx.
    """
    period         = extract_dominant_period(ctx_1d)
    period         = max(2, min(period, len(ctx_1d) // 2))
    n_full_periods = len(ctx_1d) // period

    if n_full_periods < 1:
        pattern = ctx_1d[-period:]
    else:
        patterns = [ctx_1d[i*period:(i+1)*period] for i in range(n_full_periods)]
        pattern  = np.mean(patterns, axis=0)

    n_tiles    = pred_len // period + 2
    tiled      = np.tile(pattern, n_tiles)
    offset     = len(ctx_1d) % period
    projection = tiled[offset : offset + pred_len]
    return projection.astype(np.float32)

def fft_decompose_improved(ctx_1d, period):
    """
    Remove dominant periodic components via FFT (same as fixed_experiments Exp 2.2).
    Returns (deterministic, residual).
    """
    N           = len(ctx_1d)
    X           = np.fft.rfft(ctx_1d)
    freqs       = np.fft.rfftfreq(N)
    det_mask    = np.zeros(len(X), dtype=bool)
    n_trend     = max(1, int(0.01 * len(X)))
    det_mask[:n_trend] = True
    fund_freq   = 1.0 / period
    for m in range(1, int(N / period) + 1):
        idx = np.argmin(np.abs(freqs - m * fund_freq))
        if idx < len(X):
            det_mask[max(0, idx-1):idx+2] = True
    X_det           = np.zeros_like(X)
    X_det[det_mask] = X[det_mask]
    deterministic   = np.fft.irfft(X_det, n=N)
    return deterministic, ctx_1d - deterministic

def evaluate_improved_projection(data_CT, horizon, period=144,
                                   n_windows=N_WINDOWS, label=''):
    """
    Compare vanilla vs improved-projection decomposition.
    data_CT: (C, T) RAW.
    """
    C, T      = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    if max_start <= 0:
        print(f'  [SKIP] {label}')
        return None, None

    starts = np.linspace(0, max_start, n_windows, dtype=int)
    mae_pv, mae_cv = [], []  # vanilla
    mae_pd, mae_cd = [], []  # improved decomp

    for s in starts:
        ctx_raw           = data_CT[:, s : s + CONTEXT_LEN]
        tgt_raw           = data_CT[:, s + CONTEXT_LEN : s + CONTEXT_LEN + horizon]
        ctx_norm, mu, std = instance_norm_window(ctx_raw)
        tgt_norm          = (tgt_raw - mu) / std

        # Vanilla
        mae_pv.append(mae(tgt_norm, panda_forecast(ctx_norm, horizon)))
        mae_cv.append(mae(tgt_norm, chronos_forecast(ctx_norm, horizon)))

        # Improved decomposition
        ctx_res  = np.zeros_like(ctx_raw)
        det_proj = np.zeros((C, horizon))
        for c in range(C):
            det_c, res_c    = fft_decompose_improved(ctx_raw[c], period)
            ctx_res[c]      = res_c
            det_proj[c]     = project_seasonal_improved(det_c, horizon)

        ctx_res_norm, mu_r, std_r = instance_norm_window(ctx_res)
        p_r = panda_forecast(ctx_res_norm, horizon)
        c_r = chronos_forecast(ctx_res_norm, horizon)

        p_full = ((p_r * std_r + mu_r) + det_proj - mu) / std
        c_full = ((c_r * std_r + mu_r) + det_proj - mu) / std

        mae_pd.append(mae(tgt_norm, p_full))
        mae_cd.append(mae(tgt_norm, c_full))

    def _summarise(mae_a, mae_b, tag, cond):
        diff = np.array(mae_b) - np.array(mae_a)
        try:
            _, pval = wilcoxon(diff, alternative='greater') \
                if np.any(diff != 0) else (0, 1.0)
        except Exception:
            pval = np.nan
        adv   = np.median(mae_b) - np.median(mae_a)
        iqr_a = np.percentile(mae_a,75) - np.percentile(mae_a,25)
        iqr_b = np.percentile(mae_b,75) - np.percentile(mae_b,25)
        sig   = ' *' if pval < 0.05 else (' ~' if pval < 0.10 else '')
        print(
            f'  {tag:52s}  H={horizon:4d}  '
            f'P={np.median(mae_a):.4f}[±{iqr_a:.4f}]  '
            f'C={np.median(mae_b):.4f}[±{iqr_b:.4f}]  '
            f'Adv={adv:+.4f}  p={pval:.3f}{sig}'
        )
        return {
            'label': tag, 'horizon': horizon, 'dataset': label, 'condition': cond,
            'panda_mae': np.median(mae_a), 'panda_iqr': iqr_a,
            'chronos_mae': np.median(mae_b), 'chronos_iqr': iqr_b,
            'advantage_mae': adv, 'wilcoxon_p': pval,
        }

    r_v = _summarise(mae_pv, mae_cv, f'{label}_vanilla_H{horizon}',  'vanilla')
    r_d = _summarise(mae_pd, mae_cd, f'{label}_improved_proj_H{horizon}', 'improved_proj')
    return r_v, r_d

print('Priority 5 helpers defined.')

Priority 5 helpers defined.


In [17]:
print('Priority 5: Improved Period Projection in Decomposition — Weather')
print('-' * 70)

dataset_periods = {'ETTh1': 24, 'ETTh2': 24, 'Weather': 144}
p5_results      = []

for dname, dpath in {
    'Weather': f'{DATA_DIR}/weather.csv',
    'ETTh1'  : f'{DATA_DIR}/ETTh1.csv',
    'ETTh2'  : f'{DATA_DIR}/ETTh2.csv',
}.items():
    data   = load_ts(dpath)
    period = dataset_periods[dname]
    print(f'\n  {dname}: shape={data.shape}  period={period}')

    for h in [96, 336]:
        r_v, r_d = evaluate_improved_projection(
            data, h, period=period, n_windows=N_WINDOWS, label=dname
        )
        if r_v: p5_results.append(r_v)
        if r_d: p5_results.append(r_d)

df_p5 = pd.DataFrame(p5_results)
df_p5.to_csv('p5_improved_projection_results.csv', index=False)
print('\nSaved p5_improved_projection_results.csv')

print('\n=== Priority 5 Summary ===')
for dname in ['Weather', 'ETTh1', 'ETTh2']:
    sub = df_p5[df_p5.dataset == dname]
    if sub.empty:
        continue
    van  = sub[sub.condition=='vanilla']
    imp  = sub[sub.condition=='improved_proj']
    print(f'\n  {dname}:')
    for h in [96, 336]:
        v = van[van.horizon==h]
        i = imp[imp.horizon==h]
        if len(v) and len(i):
            adv_v = float(v.advantage_mae)
            adv_i = float(i.advantage_mae)
            change = adv_i - adv_v
            print(f'    H={h}: Adv(vanilla)={adv_v:+.4f}  Adv(improved proj)={adv_i:+.4f}  delta={change:+.4f}')
            if adv_i > adv_v + 0.02:
                obs = 'Improved projection widens advantage -> residual contains genuine predictable structure.'
            elif abs(adv_i - adv_v) < 0.02:
                obs = 'Advantage unchanged -> projection quality does not affect Panda/Chronos relative performance.'
            else:
                obs = 'Improved projection narrows advantage -> some of vanilla advantage was from projection error.'
            print(f'    Observation: {obs}')

Priority 5: Improved Period Projection in Decomposition — Weather
----------------------------------------------------------------------

  Weather: shape=(21, 52696)  period=144


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Weather_vanilla_H96                                   H=  96  P=0.6128[±0.2428]  C=0.7720[±0.0661]  Adv=+0.1592  p=0.008 *
  Weather_improved_proj_H96                             H=  96  P=1.0848[±0.2152]  C=1.0468[±0.1290]  Adv=-0.0380  p=1.000


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Weather_vanilla_H336                                  H= 336  P=0.8754[±0.2712]  C=0.9974[±0.2830]  Adv=+0.1219  p=0.008 *
  Weather_improved_proj_H336                            H= 336  P=1.2408[±0.3517]  C=1.2707[±0.3479]  Adv=+0.0299  p=0.012 *

  ETTh1: shape=(7, 17420)  period=24


...

  ETTh1_vanilla_H336                                    H= 336  P=0.8429[±0.4968]  C=0.9434[±0.3993]  Adv=+0.1005  p=0.125
  ETTh1_improved_proj_H336                              H= 336  P=0.8305[±0.6071]  C=0.7923[±0.6226]  Adv=-0.0382  p=0.320

  ETTh2: shape=(7, 17420)  period=24


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  ETTh2_vanilla_H96                                     H=  96  P=0.9356[±0.4545]  C=0.9341[±0.4385]  Adv=-0.0015  p=0.371
  ETTh2_improved_proj_H96                               H=  96  P=0.8504[±0.4594]  C=0.7957[±0.4159]  Adv=-0.0546  p=0.875


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  ETTh2_vanilla_H336                                    H= 336  P=1.2494[±0.5162]  C=1.1824[±0.7886]  Adv=-0.0671  p=0.371
  ETTh2_improved_proj_H336                              H= 336  P=1.2788[±0.5435]  C=1.1362[±0.7120]  Adv=-0.1426  p=0.961

Saved p5_improved_projection_results.csv

=== Priority 5 Summary ===

  Weather:
    H=96: Adv(vanilla)=+0.1592  Adv(improved proj)=-0.0380  delta=-0.1972
    Observation: Improved projection narrows advantage -> some of vanilla advantage was from projection error.
    H=336: Adv(vanilla)=+0.1219  Adv(improved proj)=+0.0299  delta=-0.0920
    Observation: Improved projection narrows advantage -> some of vanilla advantage was from projection error.

  ETTh1:
    H=96: Adv(vanilla)=+0.0618  Adv(improved proj)=+0.0551  delta=-0.0066
    Observation: Advantage unchanged -> projection quality does not affect Panda/Chronos relative performance.
    H=336: Adv(vanilla)=+0.1005  Adv(improved proj)=-0.0382  delta=-0.1388
    Observation: Improved proje

## Final Summary Table


In [18]:
print('=' * 70)
print('ALL NEW EXPERIMENT RESULTS')
print('=' * 70)

summary_frames = [
    ('P1: Periodic Component Forecasting', df_p1),
    ('P2: Lorenz Surrogate Control',       df_p2),
    ('P3: Burgers Lambda1 + Forecast',     df_p3),
    ('P4: Subsampling Seed Variance',      df_p4),
    ('P5: Improved Period Projection',     df_p5),
]

for name, df in summary_frames:
    print(f'\n--- {name} ---')
    cols = [c for c in ['label','condition','method','nu','seed',
                         'horizon','advantage_mae','wilcoxon_p']
            if c in df.columns]
    print(df[cols].round(4).to_string(index=False))

print('\nDone. Record observations before drawing inferences.')

ALL NEW EXPERIMENT RESULTS

--- P1: Periodic Component Forecasting ---
                label     condition  horizon  advantage_mae  wilcoxon_p
     Weather_full_H96          full       96         0.1893      0.0039
 Weather_periodic_H96 periodic_only       96         0.5220      0.0039
    Weather_full_H336          full      336         0.1031      0.0195
Weather_periodic_H336 periodic_only      336         0.5014      0.0039

--- P2: Lorenz Surrogate Control ---
                        label condition  horizon  advantage_mae  wilcoxon_p
         Lorenz_rho28_chaotic   chaotic       96         0.3835      0.0039
Lorenz_surrogate_statsmatched surrogate       96         0.1715      0.3203

--- P3: Burgers Lambda1 + Forecast ---
               label    nu  horizon  advantage_mae  wilcoxon_p
Burgers_PCA_nu=2.000 2.000       96        -0.0015      0.1914
Burgers_PCA_nu=1.000 1.000       96         0.0045      0.0039
Burgers_PCA_nu=0.500 0.500       96         0.0156      0.0117
Burgers_PCA

In [19]:
print(df_p3[['nu', 'lambda1', 'advantage_mae', 'wilcoxon_p']].to_string())

      nu   lambda1  advantage_mae  wilcoxon_p
0  2.000       NaN      -0.001537    0.191406
1  1.000 -0.011359       0.004508    0.003906
2  0.500 -0.006698       0.015578    0.011719
3  0.100 -0.003614       0.100998    0.003906
4  0.050 -0.003063       0.118728    0.003906
5  0.010 -0.000998       0.151022    0.003906
6  0.005 -0.000653       0.100341    0.003906


In [20]:
print('Option A: Periodic context, real targets — Weather')
print('-' * 70)

def evaluate_periodic_real_target(data_CT, horizon, n_windows=N_WINDOWS,
                                   label='', n_harmonics=5):
    """
    Context: FFT periodic component only (top n_harmonics).
    Target:  actual future values from the series (no construction artifact).
    Normalisation: per-window on the ORIGINAL context (not the periodic context),
                   so target normalisation is consistent with vanilla evaluate().
    """
    C, T      = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    if max_start <= 0:
        print(f'  [SKIP] {label}')
        return None

    starts = np.linspace(0, max_start, n_windows, dtype=int)
    mae_p, mae_c = [], []

    for s in starts:
        ctx_raw           = data_CT[:, s : s + CONTEXT_LEN]
        tgt_raw           = data_CT[:, s + CONTEXT_LEN : s + CONTEXT_LEN + horizon]

        # Normalise using original context stats (same as vanilla evaluate)
        _, mu, std        = instance_norm_window(ctx_raw)
        tgt_norm          = (tgt_raw - mu) / std

        # Build periodic-only context, normalise with same mu/std
        ctx_per           = np.zeros_like(ctx_raw)
        for c in range(C):
            ctx_per[c]    = extract_periodic_component(ctx_raw[c], n_harmonics)
        ctx_per_norm      = (ctx_per - mu) / std

        mae_p.append(mae(tgt_norm, panda_forecast(ctx_per_norm, horizon)))
        mae_c.append(mae(tgt_norm, chronos_forecast(ctx_per_norm, horizon)))

    diff = np.array(mae_c) - np.array(mae_p)
    try:
        _, pval = wilcoxon(diff, alternative='greater') \
            if np.any(diff != 0) else (0, 1.0)
    except Exception:
        pval = np.nan

    adv   = np.median(mae_c) - np.median(mae_p)
    iqr_p = np.percentile(mae_p, 75) - np.percentile(mae_p, 25)
    iqr_c = np.percentile(mae_c, 75) - np.percentile(mae_c, 25)
    sig   = ' *' if pval < 0.05 else (' ~' if pval < 0.10 else '')

    print(
        f'  {label:50s}  H={horizon:4d}  '
        f'panda={np.median(mae_p):.4f}[±{iqr_p:.4f}]  '
        f'chronos={np.median(mae_c):.4f}[±{iqr_c:.4f}]  '
        f'Adv={adv:+.4f}  p={pval:.3f}{sig}'
    )
    return {
        'label': label, 'horizon': horizon, 'condition': 'periodic_ctx_real_tgt',
        'panda_mae': np.median(mae_p), 'panda_iqr': iqr_p,
        'chronos_mae': np.median(mae_c), 'chronos_iqr': iqr_c,
        'advantage_mae': adv, 'wilcoxon_p': pval,
    }

# Reuse vanilla results already in df_p1 for direct comparison
print('\nReference (vanilla, real targets):')
print(df_p1[df_p1.condition=='full'][['label','horizon','panda_mae','chronos_mae','advantage_mae','wilcoxon_p']].to_string(index=False))

print('\nNew (periodic context, real targets):')
opta_results = []
for h in [96, 336]:
    r = evaluate_periodic_real_target(data_weather, h,
                                       n_windows=N_WINDOWS,
                                       label=f'Weather_periodic_ctx_real_tgt_H{h}',
                                       n_harmonics=5)
    if r:
        opta_results.append(r)

df_opta = pd.DataFrame(opta_results)

print('\n=== Option A Summary ===')
print(f'{"H":>5} | {"vanilla adv":>12} | {"periodic_ctx adv":>16} | {"interpretation"}')
print('-' * 70)
for h in [96, 336]:
    van = df_p1[(df_p1.condition=='full') & (df_p1.horizon==h)]
    per = df_opta[df_opta.horizon==h]
    if len(van) and len(per):
        adv_v = float(van.advantage_mae)
        adv_p = float(per.advantage_mae)
        if adv_p > adv_v + 0.02:
            interp = 'Amplifies: periodic context helps Panda disproportionately'
        elif adv_p > 0.5 * adv_v:
            interp = 'Persists: periodic handling is a real source of advantage'
        elif adv_p > 0.05:
            interp = 'Partial: both periodic and residual contribute'
        else:
            interp = 'Collapses: advantage requires full signal'
        print(f'{h:>5} | {adv_v:>12.4f} | {adv_p:>16.4f} | {interp}')

Option A: Periodic context, real targets — Weather
----------------------------------------------------------------------

Reference (vanilla, real targets):
            label  horizon  panda_mae  chronos_mae  advantage_mae  wilcoxon_p
 Weather_full_H96       96   0.612812     0.802106       0.189294    0.003906
Weather_full_H336      336   0.875446     0.978565       0.103119    0.019531

New (periodic context, real targets):


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Weather_periodic_ctx_real_tgt_H96                   H=  96  panda=1.2007[±0.2439]  chronos=1.0473[±0.1165]  Adv=-0.1534  p=0.992


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Weather_periodic_ctx_real_tgt_H336                  H= 336  panda=1.1911[±0.2718]  chronos=1.1816[±0.2269]  Adv=-0.0095  p=0.875

=== Option A Summary ===
    H |  vanilla adv | periodic_ctx adv | interpretation
----------------------------------------------------------------------
   96 |       0.1893 |          -0.1534 | Collapses: advantage requires full signal
  336 |       0.1031 |          -0.0095 | Collapses: advantage requires full signal


In [17]:
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.signal import periodogram

print('Sensor Heterogeneity Experiment — Weather')
print('-' * 70)

def compute_channel_features(data_CT):
    """
    Compute dynamic features per channel.
    Returns (C, F) feature matrix.
    Features: std, mean abs, lag-1 autocorr, spectral entropy, dominant freq.
    """
    C, T     = data_CT.shape
    features = []
    for c in range(C):
        ts       = data_CT[c].astype(float)
        ts_norm  = (ts - ts.mean()) / (ts.std() + 1e-8)
        freqs, power = periodogram(ts_norm)
        p_norm   = power / (power.sum() + 1e-10)
        p_norm   = p_norm[p_norm > 1e-10]
        lag1_ac  = float(np.corrcoef(ts_norm[:-1], ts_norm[1:])[0, 1])
        dom_freq = float(freqs[np.argmax(power[1:]) + 1])
        features.append([
            float(ts_norm.std()),
            float(np.mean(np.abs(ts_norm))),
            lag1_ac,
            float(-np.sum(p_norm * np.log(p_norm))),
            dom_freq,
        ])
    F = np.array(features)
    # Normalise features to [0,1]
    F = (F - F.min(0)) / (F.max(0) - F.min(0) + 1e-8)
    return F

# Use full Weather series for feature computation
features_CT = compute_channel_features(data_weather)
C           = data_weather.shape[0]
print(f'Weather channels: {C}')
print(f'Feature matrix shape: {features_CT.shape}')

# Pairwise distance between channels in feature space
dist_matrix = squareform(pdist(features_CT, metric='euclidean'))
print(f'Mean inter-channel distance: {dist_matrix.mean():.4f}')
print(f'Max inter-channel distance:  {dist_matrix.max():.4f}')

# Hierarchical clustering into N_CLUSTERS groups
N_CLUSTERS = 4
Z          = linkage(features_CT, method='ward')
labels     = fcluster(Z, N_CLUSTERS, criterion='maxclust')
print(f'\nCluster assignments (4 clusters):')
for k in range(1, N_CLUSTERS+1):
    members = np.where(labels == k)[0].tolist()
    print(f'  Cluster {k}: channels {members}  (n={len(members)})')

# Build subsets
N_SUB = 7  # channels per subset (keep manageable)

def intra_cluster_heterogeneity(channel_indices, dist_matrix):
    """Mean pairwise distance within a set of channels."""
    idx = np.array(channel_indices)
    if len(idx) < 2:
        return 0.0
    sub = dist_matrix[np.ix_(idx, idx)]
    return float(sub[np.triu_indices(len(idx), k=1)].mean())

# Homogeneous subset: channels from the same cluster (pick largest cluster)
cluster_sizes = {k: (labels==k).sum() for k in range(1, N_CLUSTERS+1)}
largest_cluster = max(cluster_sizes, key=cluster_sizes.get)
homo_channels   = np.where(labels == largest_cluster)[0]
# If cluster too large, take N_SUB most central members
if len(homo_channels) > N_SUB:
    cluster_feat   = features_CT[homo_channels]
    centroid       = cluster_feat.mean(axis=0)
    dists_to_cent  = np.linalg.norm(cluster_feat - centroid, axis=1)
    homo_channels  = homo_channels[np.argsort(dists_to_cent)[:N_SUB]]
else:
    homo_channels  = homo_channels[:N_SUB]

# Heterogeneous subset: one channel from each cluster (maximally spread)
hetero_channels = []
for k in range(1, N_CLUSTERS+1):
    members = np.where(labels == k)[0]
    if len(members) > 0:
        hetero_channels.append(members[0])
# Fill to N_SUB if needed
rng = np.random.default_rng(SEED)
remaining = [c for c in range(C) if c not in hetero_channels]
while len(hetero_channels) < N_SUB:
    hetero_channels.append(remaining.pop(0))
hetero_channels = np.array(hetero_channels[:N_SUB])

homo_het  = intra_cluster_heterogeneity(homo_channels, dist_matrix)
hetero_het = intra_cluster_heterogeneity(hetero_channels, dist_matrix)

print(f'\nHomogeneous subset channels:   {homo_channels.tolist()}')
print(f'  Intra-set mean distance: {homo_het:.4f}')
print(f'Heterogeneous subset channels: {hetero_channels.tolist()}')
print(f'  Intra-set mean distance: {hetero_het:.4f}')
print(f'Heterogeneity ratio (hetero/homo): {hetero_het/(homo_het+1e-8):.2f}x')

Sensor Heterogeneity Experiment — Weather
----------------------------------------------------------------------
Weather channels: 21
Feature matrix shape: (21, 5)
Mean inter-channel distance: 0.8121
Max inter-channel distance:  1.9948

Cluster assignments (4 clusters):
  Cluster 1: channels [0, 1, 2, 3, 5, 6, 7, 8, 9, 10, 19]  (n=11)
  Cluster 2: channels [4, 12, 16, 17, 18]  (n=5)
  Cluster 3: channels [11, 20]  (n=2)
  Cluster 4: channels [13, 14, 15]  (n=3)

Homogeneous subset channels:   [9, 6, 10, 8, 5, 3, 2]
  Intra-set mean distance: 0.0353
Heterogeneous subset channels: [0, 4, 11, 13, 1, 2, 3]
  Intra-set mean distance: 0.9471
Heterogeneity ratio (hetero/homo): 26.81x


In [22]:
print('\nRunning forecast comparison on stratified subsets...')
print('-' * 70)

het_results = []

# Multiple heterogeneity levels:
# Level 1: homogeneous (same cluster)
# Level 2: mixed (two clusters)
# Level 3: heterogeneous (all clusters)

# Build mixed subset: channels from two clusters
cluster1 = np.where(labels == 1)[0]
cluster2 = np.where(labels == 2)[0]
mixed_channels = np.concatenate([
    cluster1[:max(1, N_SUB//2)],
    cluster2[:max(1, N_SUB - N_SUB//2)]
])[:N_SUB]
mixed_het = intra_cluster_heterogeneity(mixed_channels, dist_matrix)

subsets = {
    'homogeneous'  : homo_channels,
    'mixed'        : mixed_channels,
    'heterogeneous': hetero_channels,
}

for subset_name, ch_idx in subsets.items():
    data_sub = data_weather[ch_idx, :]
    het_val  = intra_cluster_heterogeneity(ch_idx, dist_matrix)
    print(f'\n  Subset: {subset_name}  (channels={ch_idx.tolist()}  het={het_val:.4f})')

    for h in [96, 336]:
        r = evaluate(data_sub, h, n_windows=N_WINDOWS,
                     label=f'Weather_{subset_name}_H{h}')
        if r:
            r['subset']          = subset_name
            r['n_channels']      = len(ch_idx)
            r['heterogeneity']   = het_val
            het_results.append(r)

df_het = pd.DataFrame(het_results)
df_het.to_csv('het_stratification_results.csv', index=False)
print('\nSaved het_stratification_results.csv')


Running forecast comparison on stratified subsets...
----------------------------------------------------------------------

  Subset: homogeneous  (channels=[9, 6, 10, 8, 5, 3, 2]  het=0.0353)


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Weather_homogeneous_H96                             H=  96  panda=0.3178[±0.2870]  chronos=0.6892[±0.2258]  Adv=+0.3714  p=0.004 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Weather_homogeneous_H336                            H= 336  panda=0.8173[±0.4989]  chronos=1.1826[±0.4878]  Adv=+0.3654  p=0.004 *

  Subset: mixed  (channels=[0, 1, 2, 4, 12, 16, 17]  het=0.6677)


...

  Weather_mixed_H336                                  H= 336  panda=0.9254[±0.2504]  chronos=0.9817[±0.2473]  Adv=+0.0563  p=0.371

  Subset: heterogeneous  (channels=[0, 4, 11, 13, 1, 2, 3]  het=0.9471)


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


  Weather_heterogeneous_H96                           H=  96  panda=0.6184[±0.2121]  chronos=0.6969[±0.2443]  Adv=+0.0785  p=0.074 ~


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Weather_heterogeneous_H336                          H= 336  panda=1.1230[±0.3600]  chronos=1.1629[±0.2913]  Adv=+0.0399  p=0.191

Saved het_stratification_results.csv


In [23]:
print('\n=== Sensor Heterogeneity Stratification Summary ===')
print(f'{"subset":>15} | {"H":>5} | {"panda_mae":>10} | {"chronos_mae":>12} | {"advantage":>10} | {"p":>7} | {"het":>6}')
print('-' * 80)

for _, row in df_het.sort_values(['horizon','heterogeneity']).iterrows():
    sig = '*' if row.wilcoxon_p < 0.05 else ''
    print(
        f'{row.subset:>15} | {row.horizon:>5} | {row.panda_mae:>10.4f} | '
        f'{row.chronos_mae:>12.4f} | {row.advantage_mae:>10.4f} | '
        f'{row.wilcoxon_p:>7.4f}{sig} | {row.heterogeneity:>6.4f}'
    )

print('\n--- Per-horizon interpretation ---')
for h in [96, 336]:
    sub = df_het[df_het.horizon == h].sort_values('heterogeneity')
    if len(sub) < 2:
        continue
    panda_maes = sub.panda_mae.values
    het_vals   = sub.heterogeneity.values
    cv         = panda_maes.std() / (panda_maes.mean() + 1e-8)
    slope, _, r, pval_r, _ = linregress(het_vals, panda_maes)

    print(f'\n  H={h}:')
    print(f'    Panda MAE across subsets: {panda_maes}')
    print(f'    CV of Panda MAE: {cv:.3f}')
    print(f'    Spearman slope (het vs panda_mae): {slope:.4f}  r={r:.3f}  p={pval_r:.3f}')

    if cv < 0.05:
        obs = 'H2 supported: Panda MAE invariant to heterogeneity. G-SWaN direction not supported by this data.'
    elif slope > 0 and pval_r < 0.10:
        obs = 'H1 supported: Panda MAE increases with heterogeneity. Sensor identity embeddings may help.'
    elif slope < 0 and pval_r < 0.10:
        obs = 'H3 supported: Panda MAE decreases with heterogeneity. Koopman embedding benefits from diverse dynamics.'
    else:
        obs = 'Inconclusive. Direction consistent with H1/H2/H3 but not significant.'
    print(f'    Observation: {obs}')

# Cross-check with P4 finding
print('\n--- Cross-check with P4 ---')
print('P4 finding: Panda MAE invariant to spatial subsampling method (CV across seeds).')
print('This experiment: Panda MAE variance across heterogeneity levels (CV across subsets).')
print('If both CVs < 0.05: strong evidence that Panda channel attention is not')
print('adapting to either subsampling method or channel heterogeneity.')


=== Sensor Heterogeneity Stratification Summary ===
         subset |     H |  panda_mae |  chronos_mae |  advantage |       p |    het
--------------------------------------------------------------------------------
    homogeneous |    96 |     0.3178 |       0.6892 |     0.3714 |  0.0039* | 0.0353
          mixed |    96 |     0.5806 |       0.7860 |     0.2054 |  0.0391* | 0.6677
  heterogeneous |    96 |     0.6184 |       0.6969 |     0.0785 |  0.0742 | 0.9471
    homogeneous |   336 |     0.8173 |       1.1826 |     0.3654 |  0.0039* | 0.0353
          mixed |   336 |     0.9254 |       0.9817 |     0.0563 |  0.3711 | 0.6677
  heterogeneous |   336 |     1.1230 |       1.1629 |     0.0399 |  0.1914 | 0.9471

--- Per-horizon interpretation ---

  H=96:
    Panda MAE across subsets: [0.31784509 0.58055243 0.61838034]
    CV of Panda MAE: 0.264
    Spearman slope (het vs panda_mae): 0.3442  r=0.982  p=0.120
    Observation: Inconclusive. Direction consistent with H1/H2/H3 but not 

In [24]:
print('Difficulty-Matched Control: Computing per-channel Chronos univariate MAE')
print('(21 channels x 8 windows = 168 forward passes, this is the slow step)')
print('-' * 70)

def per_channel_difficulty(data_CT, horizon, n_windows=N_WINDOWS):
    """
    Per-channel Chronos univariate MAE.
    Returns array (C,) — mean MAE per channel across windows.
    """
    C, T      = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    starts    = np.linspace(0, max_start, n_windows, dtype=int)
    ch_maes   = np.zeros(C)

    for c in range(C):
        maes = []
        for s in starts:
            ctx_raw           = data_CT[c:c+1, s : s+CONTEXT_LEN]
            tgt_raw           = data_CT[c:c+1, s+CONTEXT_LEN : s+CONTEXT_LEN+horizon]
            ctx_norm, mu, std = instance_norm_window(ctx_raw)
            tgt_norm          = (tgt_raw - mu) / std
            pred              = chronos_forecast(ctx_norm, horizon)
            maes.append(mae(tgt_norm, pred))
        ch_maes[c] = np.mean(maes)
        print(f'  Channel {c:2d}: Chronos MAE = {ch_maes[c]:.4f}')

    return ch_maes

difficulty = per_channel_difficulty(data_weather, 96, n_windows=N_WINDOWS)

print(f'\nDifficulty range: [{difficulty.min():.4f}, {difficulty.max():.4f}]')
print(f'Mean: {difficulty.mean():.4f}  Std: {difficulty.std():.4f}')
print(f'\nDifficulty by cluster:')
for k in range(1, N_CLUSTERS+1):
    members = np.where(labels == k)[0]
    print(f'  Cluster {k}: mean={difficulty[members].mean():.4f}  '
          f'range=[{difficulty[members].min():.4f}, {difficulty[members].max():.4f}]  '
          f'channels={members.tolist()}')

We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


Difficulty-Matched Control: Computing per-channel Chronos univariate MAE
(21 channels x 8 windows = 168 forward passes, this is the slow step)
----------------------------------------------------------------------


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Channel  0: Chronos MAE = 0.4135


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

...

  Channel 18: Chronos MAE = 0.6885


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Channel 19: Chronos MAE = 0.6440


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


  Channel 20: Chronos MAE = 0.9268

Difficulty range: [0.2682, 1.0969]
Mean: 0.7433  Std: 0.1870

Difficulty by cluster:
  Cluster 1: mean=0.7087  range=[0.4135, 0.9733]  channels=[0, 1, 2, 3, 5, 6, 7, 8, 9, 10, 19]
  Cluster 2: mean=0.8106  range=[0.6885, 0.9479]  channels=[4, 12, 16, 17, 18]
  Cluster 3: mean=1.0118  range=[0.9268, 1.0969]  channels=[11, 20]
  Cluster 4: mean=0.5786  range=[0.2682, 0.7944]  channels=[13, 14, 15]


In [25]:
def build_difficulty_matched_subset(candidate_channels, target_mean,
                                     difficulty, n_channels,
                                     n_trials=1000, seed=SEED):
    """
    Random search over subsets of candidate_channels to find
    n_channels whose mean difficulty is closest to target_mean.
    Returns (best_subset, achieved_mean_difficulty).
    """
    rng        = np.random.default_rng(seed)
    best_sub   = None
    best_delta = np.inf

    for _ in range(n_trials):
        if len(candidate_channels) <= n_channels:
            sub = candidate_channels.copy()
        else:
            sub = rng.choice(candidate_channels, size=n_channels, replace=False)
        delta = abs(difficulty[sub].mean() - target_mean)
        if delta < best_delta:
            best_delta = delta
            best_sub   = sub.copy()

    return best_sub, float(difficulty[best_sub].mean())

# Reference: heterogeneous subset difficulty
target_difficulty = difficulty[hetero_channels].mean()
print(f'Target mean difficulty (heterogeneous subset): {target_difficulty:.4f}')

# Homo matched: pool = all channels in largest cluster
homo_pool = np.where(labels == largest_cluster)[0]
homo_matched, homo_matched_diff = build_difficulty_matched_subset(
    homo_pool, target_difficulty, difficulty, N_SUB, n_trials=1000
)

# Mixed matched: pool = channels from two clusters
mixed_pool = np.concatenate([
    np.where(labels == 1)[0],
    np.where(labels == 2)[0]
])
mixed_matched, mixed_matched_diff = build_difficulty_matched_subset(
    mixed_pool, target_difficulty, difficulty, N_SUB, n_trials=1000
)

homo_matched_het  = intra_cluster_heterogeneity(homo_matched,  dist_matrix)
mixed_matched_het = intra_cluster_heterogeneity(mixed_matched, dist_matrix)
hetero_het_val    = intra_cluster_heterogeneity(hetero_channels, dist_matrix)

print(f'\nDifficulty-matched subsets:')
print(f'  {"Subset":>15} | {"channels":>35} | {"het":>6} | {"mean_diff":>10} | {"delta_from_target":>18}')
print('-' * 95)
for name, ch, h_val, d_val in [
    ('homo_matched',  homo_matched,   homo_matched_het,  homo_matched_diff),
    ('mixed_matched', mixed_matched,  mixed_matched_het, mixed_matched_diff),
    ('heterogeneous', hetero_channels, hetero_het_val,   target_difficulty),
]:
    delta = abs(d_val - target_difficulty)
    print(f'  {name:>15} | {str(ch.tolist()):>35} | {h_val:>6.4f} | {d_val:>10.4f} | {delta:>18.4f}')

# Warn if difficulty matching is poor
max_delta = max(abs(homo_matched_diff - target_difficulty),
                abs(mixed_matched_diff - target_difficulty))
if max_delta > 0.05:
    print(f'\nWARNING: Difficulty matching is imperfect (max delta={max_delta:.4f}). '
          f'Interpret results with caution — confound may not be fully controlled.')
else:
    print(f'\nDifficulty matching acceptable (max delta={max_delta:.4f}).')

Target mean difficulty (heterogeneous subset): 0.8389

Difficulty-matched subsets:
           Subset |                            channels |    het |  mean_diff |  delta_from_target
-----------------------------------------------------------------------------------------------
     homo_matched |               [6, 8, 5, 3, 7, 2, 1] | 0.0857 |     0.7996 |             0.0392
    mixed_matched |             [1, 3, 8, 4, 16, 2, 17] | 0.6021 |     0.8393 |             0.0004
    heterogeneous |             [0, 4, 11, 13, 1, 2, 3] | 0.9471 |     0.8389 |             0.0000

Difficulty matching acceptable (max delta=0.0392).


In [26]:
print('Running forecast comparison on difficulty-matched subsets...')
print('-' * 70)

dm_results = []
dm_subsets = {
    'homo_matched'  : (homo_matched,   homo_matched_het,  homo_matched_diff),
    'mixed_matched' : (mixed_matched,  mixed_matched_het, mixed_matched_diff),
    'heterogeneous' : (hetero_channels, hetero_het_val,   target_difficulty),
}

for subset_name, (ch_idx, het_val, diff_val) in dm_subsets.items():
    data_sub = data_weather[ch_idx, :]
    print(f'\n  {subset_name}  het={het_val:.4f}  mean_difficulty={diff_val:.4f}')
    for h in [96, 336]:
        r = evaluate(data_sub, h, n_windows=N_WINDOWS,
                     label=f'DM_{subset_name}_H{h}')
        if r:
            r['subset']          = subset_name
            r['heterogeneity']   = het_val
            r['mean_difficulty'] = diff_val
            dm_results.append(r)

df_dm = pd.DataFrame(dm_results)
df_dm.to_csv('difficulty_matched_results.csv', index=False)
print('\nSaved difficulty_matched_results.csv')

print('\n=== Difficulty-Matched Summary ===')
print(f'{"subset":>15} | {"H":>5} | {"het":>6} | {"difficulty":>10} | '
      f'{"panda_mae":>10} | {"chronos_mae":>11} | {"advantage":>10} | {"p":>7}')
print('-' * 88)
for _, row in df_dm.sort_values(['horizon','heterogeneity']).iterrows():
    sig = '*' if row.wilcoxon_p < 0.05 else ''
    print(f'{row.subset:>15} | {int(row.horizon):>5} | {row.heterogeneity:>6.4f} | '
          f'{row.mean_difficulty:>10.4f} | {row.panda_mae:>10.4f} | '
          f'{row.chronos_mae:>11.4f} | {row.advantage_mae:>10.4f} | '
          f'{row.wilcoxon_p:>6.4f}{sig}')

print('\n--- Interpretation ---')
for h in [96, 336]:
    sub        = df_dm[df_dm.horizon == h].sort_values('heterogeneity')
    p_maes     = sub.panda_mae.values
    diffs      = sub.mean_difficulty.values
    hets       = sub.heterogeneity.values
    diff_cv    = diffs.std() / (diffs.mean() + 1e-8)
    panda_cv   = p_maes.std() / (p_maes.mean() + 1e-8)

    print(f'\n  H={h}:')
    print(f'    Difficulty CV across subsets: {diff_cv:.3f}')
    print(f'    Panda MAE CV across subsets:  {panda_cv:.3f}')

    if diff_cv < 0.10 and panda_cv > 0.05:
        obs = ('Difficulty well-matched (CV<10%) but Panda MAE still varies (CV>5%). '
               'Heterogeneity effect on Panda is NOT explained by individual channel difficulty. '
               'H1 (sensor heterogeneity is the bottleneck) is supported.')
    elif diff_cv < 0.10 and panda_cv <= 0.05:
        obs = ('Difficulty well-matched and Panda MAE is stable. '
               'Heterogeneity does not affect Panda after difficulty control. '
               'H2 (Panda is robust to heterogeneity) is supported.')
    else:
        obs = (f'Difficulty not well-matched (CV={diff_cv:.3f}). '
               'Confound not fully controlled. Result is inconclusive.')
    print(f'    Observation: {obs}')

# Core question: does Panda advantage still drop monotonically with heterogeneity
# after difficulty matching?
print('\n--- Core question: does advantage still drop with heterogeneity? ---')
for h in [96, 336]:
    sub  = df_dm[df_dm.horizon == h].sort_values('heterogeneity')
    advs = sub.advantage_mae.values
    hets = sub.heterogeneity.values
    slope, _, r, _, _ = linregress(hets, advs)
    print(f'  H={h}: Advantage = {advs}')
    print(f'         Slope = {slope:.4f}  r = {r:.3f}')
    if slope < -0.1 and r < -0.8:
        print(f'         Advantage still drops with heterogeneity after difficulty matching.')
    elif abs(slope) < 0.1:
        print(f'         Advantage is flat after difficulty matching. '
              f'Original drop was a difficulty confound.')
    else:
        print(f'         Weak or ambiguous trend.')

Running forecast comparison on difficulty-matched subsets...
----------------------------------------------------------------------

  homo_matched  het=0.0857  mean_difficulty=0.7996


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  DM_homo_matched_H96                                 H=  96  panda=0.3306[±0.1773]  chronos=0.6997[±0.2332]  Adv=+0.3691  p=0.004 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  DM_homo_matched_H336                                H= 336  panda=0.8413[±0.5162]  chronos=1.1467[±0.5096]  Adv=+0.3054  p=0.004 *

  mixed_matched  het=0.6021  mean_difficulty=0.8393


...

  DM_mixed_matched_H336                               H= 336  panda=0.8562[±0.4587]  chronos=0.9465[±0.2629]  Adv=+0.0903  p=0.020 *

  heterogeneous  het=0.9471  mean_difficulty=0.8389


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  DM_heterogeneous_H96                                H=  96  panda=0.6184[±0.2121]  chronos=0.8362[±0.1950]  Adv=+0.2178  p=0.074 ~


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  DM_heterogeneous_H336                               H= 336  panda=1.1230[±0.3600]  chronos=1.2429[±0.4718]  Adv=+0.1199  p=0.125

Saved difficulty_matched_results.csv

=== Difficulty-Matched Summary ===
         subset |     H |    het | difficulty |  panda_mae | chronos_mae |  advantage |       p
----------------------------------------------------------------------------------------
   homo_matched |    96 | 0.0857 |     0.7996 |     0.3306 |      0.6997 |     0.3691 | 0.0039*
  mixed_matched |    96 | 0.6021 |     0.8393 |     0.4749 |      0.8083 |     0.3334 | 0.0039*
  heterogeneous |    96 | 0.9471 |     0.8389 |     0.6184 |      0.8362 |     0.2178 | 0.0742
   homo_matched |   336 | 0.0857 |     0.7996 |     0.8413 |      1.1467 |     0.3054 | 0.0039*
  mixed_matched |   336 | 0.6021 |     0.8393 |     0.8562 |      0.9465 |     0.0903 | 0.0195*
  heterogeneous |   336 | 0.9471 |     0.8389 |     1.1230 |      1.2429 |     0.1199 | 0.1250

--- Interpretation ---

  H=96:
   

In [27]:
print('Chronos-Only Heterogeneity Sensitivity')
print('Question: Does Chronos MAE also vary with heterogeneity, or is degradation Panda-specific?')
print('-' * 70)

def evaluate_chronos_only(data_CT, horizon, n_windows=N_WINDOWS, label=''):
    """
    Chronos univariate only — each channel independently.
    Returns per-channel and aggregate MAE.
    """
    C, T      = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    if max_start <= 0:
        print(f'  [SKIP] {label}')
        return None

    starts   = np.linspace(0, max_start, n_windows, dtype=int)
    mae_c    = []

    for s in starts:
        ctx_raw           = data_CT[:, s : s + CONTEXT_LEN]
        tgt_raw           = data_CT[:, s + CONTEXT_LEN : s + CONTEXT_LEN + horizon]
        ctx_norm, mu, std = instance_norm_window(ctx_raw)
        tgt_norm          = (tgt_raw - mu) / std
        mae_c.append(mae(tgt_norm, chronos_forecast(ctx_norm, horizon)))

    med   = float(np.median(mae_c))
    iqr   = float(np.percentile(mae_c,75) - np.percentile(mae_c,25))
    print(f'  {label:50s}  H={horizon:4d}  chronos={med:.4f}[±{iqr:.4f}]')
    return {'label': label, 'horizon': horizon,
            'chronos_mae': med, 'chronos_iqr': iqr}

# Run Chronos only on all three difficulty-matched subsets
subsets_dm = {
    'homo_matched'  : (homo_matched,    0.0857, 0.7996),
    'mixed_matched' : (mixed_matched,   0.6021, 0.8393),
    'heterogeneous' : (hetero_channels, 0.9471, 0.8389),
}

chronos_het_results = []
for subset_name, (ch_idx, het_val, diff_val) in subsets_dm.items():
    data_sub = data_weather[ch_idx, :]
    for h in [96, 336]:
        r = evaluate_chronos_only(data_sub, h, n_windows=N_WINDOWS,
                                   label=f'Chronos_{subset_name}_H{h}')
        if r:
            r['subset']        = subset_name
            r['heterogeneity'] = het_val
            r['difficulty']    = diff_val
            chronos_het_results.append(r)

df_ch = pd.DataFrame(chronos_het_results)

We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


Chronos-Only Heterogeneity Sensitivity
Question: Does Chronos MAE also vary with heterogeneity, or is degradation Panda-specific?
----------------------------------------------------------------------


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Chronos_homo_matched_H96                            H=  96  chronos=0.7757[±0.2795]


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

...

  Chronos_mixed_matched_H336                          H= 336  chronos=0.8675[±0.6415]


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Chronos_heterogeneous_H96                           H=  96  chronos=0.6869[±0.2868]


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


  Chronos_heterogeneous_H336                          H= 336  chronos=1.0887[±0.4174]


In [28]:
print('\n=== Chronos vs Panda Heterogeneity Sensitivity ===')
print(f'{"subset":>15} | {"H":>5} | {"het":>6} | {"diff":>6} | '
      f'{"panda_mae":>10} | {"chronos_mae":>12} | {"panda_CV_note"}')
print('-' * 90)

# Pull Panda MAEs from difficulty-matched experiment
panda_dm = {
    ('homo_matched',  96) : 0.3306,
    ('mixed_matched', 96) : 0.4749,
    ('heterogeneous', 96) : 0.6184,
    ('homo_matched',  336): 0.8413,
    ('mixed_matched', 336): 0.8562,
    ('heterogeneous', 336): 1.1230,
}

for _, row in df_ch.sort_values(['horizon','heterogeneity']).iterrows():
    p_mae = panda_dm.get((row.subset, int(row.horizon)), np.nan)
    print(f'{row.subset:>15} | {int(row.horizon):>5} | {row.heterogeneity:>6.4f} | '
          f'{row.difficulty:>6.4f} | {p_mae:>10.4f} | {row.chronos_mae:>12.4f}')

print('\n--- CV comparison ---')
for h in [96, 336]:
    sub     = df_ch[df_ch.horizon == h].sort_values('heterogeneity')
    c_maes  = sub.chronos_mae.values
    p_maes  = np.array([panda_dm[(s, h)] for s in
                        sub.sort_values('heterogeneity').subset.values])
    c_cv    = c_maes.std() / (c_maes.mean() + 1e-8)
    p_cv    = p_maes.std() / (p_maes.mean() + 1e-8)
    c_slope, _, c_r, _, _ = linregress(sub.heterogeneity.values, c_maes)
    p_slope, _, p_r, _, _ = linregress(sub.heterogeneity.values, p_maes)

    print(f'\n  H={h}:')
    print(f'    Panda   MAE: {p_maes}  CV={p_cv:.3f}  slope={p_slope:.4f}  r={p_r:.3f}')
    print(f'    Chronos MAE: {c_maes}  CV={c_cv:.3f}  slope={c_slope:.4f}  r={c_r:.3f}')

    if p_cv > 0.10 and c_cv < 0.05:
        obs = ('Panda MAE varies with heterogeneity; Chronos MAE is stable. '
               'Degradation is PANDA-SPECIFIC. '
               'Strong support for architectural bottleneck (H1).')
    elif p_cv > 0.10 and c_cv > 0.05:
        ratio = p_cv / (c_cv + 1e-8)
        if ratio > 2.0:
            obs = (f'Both models vary, but Panda CV ({p_cv:.3f}) is {ratio:.1f}x '
                   f'Chronos CV ({c_cv:.3f}). Panda disproportionately sensitive. '
                   'Partial support for H1; multivariate difficulty confound not ruled out.')
        else:
            obs = (f'Both models vary similarly (Panda CV={p_cv:.3f}, '
                   f'Chronos CV={c_cv:.3f}). Heterogeneity affects both. '
                   'H2 (multivariate difficulty) is the more likely explanation.')
    else:
        obs = 'Neither model varies substantially. Heterogeneity is not the driver.'
    print(f'    Observation: {obs}')

print('\n--- Key diagnostic: mixed vs heterogeneous (difficulty-identical pair) ---')
for h in [96, 336]:
    c_mixed = float(df_ch[(df_ch.subset=='mixed_matched') &
                           (df_ch.horizon==h)].chronos_mae)
    c_hetero = float(df_ch[(df_ch.subset=='heterogeneous') &
                            (df_ch.horizon==h)].chronos_mae)
    p_mixed  = panda_dm[('mixed_matched', h)]
    p_hetero = panda_dm[('heterogeneous', h)]

    print(f'\n  H={h} (difficulty matched: 0.8393 vs 0.8389):')
    print(f'    Panda:   mixed={p_mixed:.4f}  hetero={p_hetero:.4f}  '
          f'delta={p_hetero-p_mixed:+.4f}  '
          f'rel_change={100*(p_hetero-p_mixed)/p_mixed:+.1f}%')
    print(f'    Chronos: mixed={c_mixed:.4f}  hetero={c_hetero:.4f}  '
          f'delta={c_hetero-c_mixed:+.4f}  '
          f'rel_change={100*(c_hetero-c_mixed)/c_mixed:+.1f}%')

    if abs(p_hetero - p_mixed) > 2 * abs(c_hetero - c_mixed):
        print(f'    Panda delta is >2x Chronos delta. '
              f'Heterogeneity effect is disproportionately on Panda.')
    elif abs(c_hetero - c_mixed) > 2 * abs(p_hetero - p_mixed):
        print(f'    Chronos delta is >2x Panda delta. '
              f'Heterogeneity effect is disproportionately on Chronos.')
    else:
        print(f'    Panda and Chronos affected similarly. '
              f'Effect is not model-specific.')

df_ch.to_csv('chronos_heterogeneity_results.csv', index=False)
print('\nSaved chronos_heterogeneity_results.csv')


=== Chronos vs Panda Heterogeneity Sensitivity ===
         subset |     H |    het |   diff |  panda_mae |  chronos_mae | panda_CV_note
------------------------------------------------------------------------------------------
   homo_matched |    96 | 0.0857 | 0.7996 |     0.3306 |       0.7757
  mixed_matched |    96 | 0.6021 | 0.8393 |     0.4749 |       0.8167
  heterogeneous |    96 | 0.9471 | 0.8389 |     0.6184 |       0.6869
   homo_matched |   336 | 0.0857 | 0.7996 |     0.8413 |       1.0771
  mixed_matched |   336 | 0.6021 | 0.8393 |     0.8562 |       0.8675
  heterogeneous |   336 | 0.9471 | 0.8389 |     1.1230 |       1.0887

--- CV comparison ---

  H=96:
    Panda   MAE: [0.3306 0.4749 0.6184]  CV=0.248  slope=0.3298  r=0.994
    Chronos MAE: [0.77571827 0.81671935 0.68694195]  CV=0.071  slope=-0.0887  r=-0.580
    Observation: Both models vary, but Panda CV (0.248) is 3.5x Chronos CV (0.071). Panda disproportionately sensitive. Partial support for H1; multivariate di

In [10]:
# EXP 20: Does decomposition hurt Chronos as much as Panda?
# Reuses fft_decompose_improved and project_seasonal_improved from P5 cells.
# No new model architecture needed.
data_weather = load_ts(f'{DATA_DIR}/weather.csv')
print('Exp 20: Chronos Residual Ablation on Weather')
print('-' * 70)

def evaluate_chronos_decomp(data_CT, horizon, period=144,
                             n_windows=N_WINDOWS, label=''):
    """
    Run Chronos on:
      (a) vanilla full signal
      (b) FFT residual only (improved projection, same as P5)
    Both evaluated against real targets.
    Returns per-condition median MAE and wilcoxon p.
    """
    C, T      = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    if max_start <= 0:
        print(f'  [SKIP] {label}')
        return None

    starts = np.linspace(0, max_start, n_windows, dtype=int)
    mae_van, mae_res = [], []

    for s in starts:
        ctx_raw           = data_CT[:, s : s + CONTEXT_LEN]
        tgt_raw           = data_CT[:, s + CONTEXT_LEN : s + CONTEXT_LEN + horizon]
        ctx_norm, mu, std = instance_norm_window(ctx_raw)
        tgt_norm          = (tgt_raw - mu) / std

        # Vanilla Chronos
        mae_van.append(mae(tgt_norm, chronos_forecast(ctx_norm, horizon)))

        # Residual Chronos (same decomp as P5)
        ctx_res  = np.zeros_like(ctx_raw)
        det_proj = np.zeros((C, horizon), dtype=np.float32)
        for c in range(C):
            det_c, res_c = fft_decompose_improved(ctx_raw[c], period)
            ctx_res[c]   = res_c
            det_proj[c]  = project_seasonal_improved(det_c, horizon)

        ctx_res_norm, mu_r, std_r = instance_norm_window(ctx_res)
        c_res   = chronos_forecast(ctx_res_norm, horizon)
        c_full  = ((c_res * std_r + mu_r) + det_proj - mu) / std
        mae_res.append(mae(tgt_norm, c_full))

    diff_van_res = np.array(mae_van) - np.array(mae_res)
    try:
        _, p_van_res = wilcoxon(diff_van_res) \
            if np.any(diff_van_res != 0) else (0, 1.0)
    except Exception:
        p_van_res = np.nan

    med_van = np.median(mae_van)
    med_res = np.median(mae_res)
    iqr_van = np.percentile(mae_van,75) - np.percentile(mae_van,25)
    iqr_res = np.percentile(mae_res,75) - np.percentile(mae_res,25)
    delta   = med_res - med_van  # positive = decomp hurts Chronos

    print(f'  {label}  H={horizon}')
    print(f'    Chronos vanilla:  {med_van:.4f} [±{iqr_van:.4f}]')
    print(f'    Chronos residual: {med_res:.4f} [±{iqr_res:.4f}]')
    print(f'    Delta (res-van):  {delta:+.4f}  p={p_van_res:.3f}')
    return {
        'label': label, 'horizon': horizon,
        'chronos_vanilla': med_van, 'chronos_residual': med_res,
        'delta': delta, 'p': p_van_res,
    }

exp20_results = []
for h in [96, 336]:
    r = evaluate_chronos_decomp(data_weather, h, period=144,
                                 n_windows=N_WINDOWS,
                                 label=f'Weather_H{h}')
    if r:
        exp20_results.append(r)

df_exp20 = pd.DataFrame(exp20_results)
df_exp20.to_csv('exp20_chronos_residual.csv', index=False)

# Pull Panda decomp deltas from P5 for comparison
# P5 Panda vanilla/improved from df_p5
panda_deltas = {96: -0.0380 - 0.1592, 336: 0.0299 - 0.1219}  # adv_improved - adv_vanilla

print('\n=== Exp 20 Summary ===')
print(f'{"H":>5} | {"Chronos delta":>14} | {"Panda delta (P5)":>16} | Interpretation')
print('-' * 65)
for _, row in df_exp20.iterrows():
    h      = int(row.horizon)
    p_delta = panda_deltas.get(h, np.nan)
    if abs(row.delta) < 0.02 and abs(p_delta) > 0.05:
        interp = 'Chronos stable, Panda degrades -> Panda-specific. Supports H1.'
    elif row.delta > 0.05 and p_delta > 0.05:
        interp = 'Both degrade similarly -> projection error or signal difficulty. H2/H3.'
    elif row.delta < -0.05:
        interp = 'Decomp helps Chronos -> Chronos was hurt by periodicity. H2.'
    else:
        interp = 'Ambiguous.'
    print(f'{h:>5} | {row.delta:>14.4f} | {p_delta:>16.4f} | {interp}')

We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


Exp 20: Chronos Residual Ablation on Weather
----------------------------------------------------------------------


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Weather_H96  H=96
    Chronos vanilla:  0.7695 [±0.0776]
    Chronos residual: 1.0464 [±0.1812]
    Delta (res-van):  +0.2770  p=0.016


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Weather_H336  H=336
    Chronos vanilla:  1.0012 [±0.2869]
    Chronos residual: 1.2461 [±0.3728]
    Delta (res-van):  +0.2448  p=0.008

=== Exp 20 Summary ===
    H |  Chronos delta | Panda delta (P5) | Interpretation
-----------------------------------------------------------------
   96 |         0.2770 |          -0.1972 | Ambiguous.
  336 |         0.2448 |          -0.0920 | Ambiguous.


In [11]:
# EXP 21: Does signal complexity (permutation entropy) predict Panda advantage?
# Tests Koopman lifting hypothesis: Panda benefits from nonlinear signals.
# Pure analysis — no new model runs.

print('Exp 21: Permutation Entropy as Panda Advantage Predictor')
print('-' * 70)

def permutation_entropy(series, order=3, delay=1, normalise=True):
    """
    Permutation entropy of a 1D time series.
    order: embedding dimension (3-7 typical)
    delay: time delay
    normalise: divide by log(order!) to get [0,1]
    """
    from itertools import permutations
    from math import factorial, log2

    N       = len(series)
    n_embed = N - (order - 1) * delay
    if n_embed < 10:
        return np.nan

    # Build ordinal patterns
    patterns = []
    for i in range(n_embed):
        embed = series[i : i + order * delay : delay]
        rank  = np.argsort(np.argsort(embed))
        patterns.append(tuple(rank))

    # Count pattern frequencies
    from collections import Counter
    counts = Counter(patterns)
    total  = sum(counts.values())
    probs  = np.array([v / total for v in counts.values()])
    probs  = probs[probs > 0]

    pe = -np.sum(probs * np.log2(probs))
    if normalise:
        pe /= log2(factorial(order))
    return float(pe)

def mean_pe_series(series_2d, order=3, delay=1):
    """Mean permutation entropy across channels (C, T) array."""
    pes = [permutation_entropy(series_2d[c], order, delay)
           for c in range(series_2d.shape[0])]
    return float(np.nanmean(pes))

# Collect (PE, Panda_advantage) pairs from all existing results

records = []

# --- Weather (from Exp 8, n=20) ---
for h, adv in [(96, 0.174), (192, 0.236), (336, 0.236)]:
    pe = mean_pe_series(data_weather, order=3)
    records.append({'system': 'Weather', 'horizon': h,
                    'pe': pe, 'advantage': adv, 'source': 'Exp8'})

# --- ETTh1 (from Exp 8) ---
data_ett1 = load_ts(f'{DATA_DIR}/ETTh1.csv')
pe_ett1   = mean_pe_series(data_ett1, order=3)
for h, adv in [(96, -0.064), (192, -0.036), (336, 0.044), (720, 0.027)]:
    records.append({'system': 'ETTh1', 'horizon': h,
                    'pe': pe_ett1, 'advantage': adv, 'source': 'Exp8'})

# --- ETTh2 (from Exp 8) ---
data_ett2 = load_ts(f'{DATA_DIR}/ETTh2.csv')
pe_ett2   = mean_pe_series(data_ett2, order=3)
for h, adv in [(96, 0.076), (192, -0.019), (336, 0.185), (720, -0.011)]:
    records.append({'system': 'ETTh2', 'horizon': h,
                    'pe': pe_ett2, 'advantage': adv, 'source': 'Exp8'})

# --- Lorenz rho sweep (from Exp 3, approximate values) ---
lorenz_results = {
    10: (0, -0.29),  20: (0.22, 0.20),
    24: (0.34, 0.33), 28: (0.64, 1.66), 60: (0.69, 1.63)
}
for rho, (adv, _) in lorenz_results.items():
    lor = simulate_lorenz(n_steps=3000, rho=float(rho))
    pe  = permutation_entropy(lor, order=3)
    records.append({'system': f'Lorenz_rho{rho}', 'horizon': 96,
                    'pe': pe, 'advantage': adv, 'source': 'Exp3'})

# --- Burgers viscosity sweep (from Exp 10, H=128) ---
burgers_adv = {
    2.0: 0.004, 1.0: 0.038, 0.5: 0.062,
    0.1: 0.111, 0.05: 0.149, 0.01: 0.095, 0.005: 0.122
}
for nu, adv in burgers_adv.items():
    print(f'  Simulating Burgers nu={nu} for PE...')
    U   = simulate_burgers_stable(T=500, N_x=64, nu=nu)
    pe  = mean_pe_series(U.T[:8], order=3)  # first 8 spatial locations
    records.append({'system': f'Burgers_nu{nu}', 'horizon': 128,
                    'pe': pe, 'advantage': adv, 'source': 'Exp10'})

df_pe = pd.DataFrame(records)
df_pe.to_csv('exp21_permutation_entropy.csv', index=False)

# Spearman correlation
from scipy.stats import spearmanr
rho_all, p_all   = spearmanr(df_pe.pe, df_pe.advantage)
# Exclude ETT (horizon-varying) — use median per system
df_sys    = df_pe.groupby('system').agg({'pe':'mean','advantage':'mean'}).reset_index()
rho_sys, p_sys = spearmanr(df_sys.pe, df_sys.advantage)

print('\n=== Exp 21 Summary ===')
print(f'{"system":>20} | {"PE":>6} | {"mean_adv":>9}')
print('-' * 45)
for _, row in df_sys.sort_values('pe').iterrows():
    print(f'{row.system:>20} | {row.pe:>6.3f} | {row.advantage:>9.4f}')

print(f'\nSpearman (all windows):     r={rho_all:.3f}  p={p_all:.3f}')
print(f'Spearman (per system mean): r={rho_sys:.3f}  p={p_sys:.3f}')

if rho_sys > 0.6 and p_sys < 0.10:
    obs = ('Positive Spearman r > 0.6. PE predicts Panda advantage. '
           'Koopman lifting hypothesis supported: Panda benefits from nonlinear/complex signals.')
elif abs(rho_sys) < 0.3:
    obs = ('Near-zero correlation. PE does not predict advantage. '
           'Koopman lifting hypothesis not supported by this measure.')
else:
    obs = f'Moderate correlation (r={rho_sys:.3f}). Suggestive but not conclusive.'
print(f'Observation: {obs}')

Exp 21: Permutation Entropy as Panda Advantage Predictor
----------------------------------------------------------------------
  Simulating Burgers nu=2.0 for PE...
  Simulating Burgers nu=1.0 for PE...
  Simulating Burgers nu=0.5 for PE...
  Simulating Burgers nu=0.1 for PE...
  Simulating Burgers nu=0.05 for PE...
  Simulating Burgers nu=0.01 for PE...
  Simulating Burgers nu=0.005 for PE...

=== Exp 21 Summary ===
              system |     PE |  mean_adv
---------------------------------------------
       Burgers_nu2.0 |  0.024 |    0.0040
       Burgers_nu1.0 |  0.039 |    0.0380
     Burgers_nu0.005 |  0.054 |    0.1220
       Burgers_nu0.5 |  0.062 |    0.0620
      Burgers_nu0.01 |  0.069 |    0.0950
      Burgers_nu0.05 |  0.071 |    0.1490
       Burgers_nu0.1 |  0.132 |    0.1110
        Lorenz_rho28 |  0.460 |    0.6400
        Lorenz_rho20 |  0.466 |    0.2200
        Lorenz_rho24 |  0.470 |    0.3400
        Lorenz_rho60 |  0.473 |    0.6900
        Lorenz_rho10 |  0.49

In [12]:
# EXP 19: Synthetic complexity continuum
# Harmonic oscillator → Van der Pol → Duffing → Rossler → Lorenz
# Tests where Panda advantage appears as dynamical complexity increases.

print('Exp 19: Synthetic Complexity Continuum')
print('-' * 70)

def simulate_harmonic(n_steps=3000, omega=1.0, seed=SEED):
    """Simple harmonic oscillator x'' + omega^2 x = 0."""
    rng = np.random.default_rng(seed)
    dt  = 0.05
    x, v = float(rng.standard_normal()), float(rng.standard_normal())
    traj = []
    for _ in range(n_steps):
        traj.append(x)
        x_new = x + v * dt
        v_new = v - omega**2 * x * dt
        x, v  = x_new, v_new
    return np.array(traj, dtype=np.float32)

def simulate_vanderpol(n_steps=3000, mu=2.0, seed=SEED):
    """Van der Pol oscillator: nonlinear limit cycle."""
    rng = np.random.default_rng(seed)
    def vdp(t, y):
        return [y[1], mu*(1 - y[0]**2)*y[1] - y[0]]
    ic  = rng.standard_normal(2).tolist()
    sol = solve_ivp(vdp, [0, n_steps*0.05], ic,
                    t_eval=np.linspace(0, n_steps*0.05, n_steps),
                    method='RK45', rtol=1e-8, atol=1e-8)
    return sol.y[0].astype(np.float32)

def simulate_duffing(n_steps=3000, delta=0.3, alpha=-1.0,
                     beta=1.0, gamma=0.37, omega=1.2, seed=SEED):
    """Duffing oscillator: nonlinear, weakly chaotic at these params."""
    rng = np.random.default_rng(seed)
    dt  = 2*np.pi / omega / 50
    x, v = float(rng.standard_normal()), float(rng.standard_normal())
    traj = []
    t    = 0.0
    for _ in range(n_steps):
        traj.append(x)
        ax    = -delta*v - alpha*x - beta*x**3 + gamma*np.cos(omega*t)
        x_new = x + v*dt
        v_new = v + ax*dt
        x, v, t = x_new, v_new, t+dt
    return np.array(traj, dtype=np.float32)

def simulate_rossler(n_steps=3000, a=0.2, b=0.2, c=5.7, seed=SEED):
    """Rossler attractor: chaotic but simpler than Lorenz."""
    rng = np.random.default_rng(seed)
    def rhs(t, y):
        return [-y[1]-y[2], y[0]+a*y[1], b+y[2]*(y[0]-c)]
    ic  = rng.standard_normal(3).tolist()
    sol = solve_ivp(rhs, [0, n_steps*0.05], ic,
                    t_eval=np.linspace(0, n_steps*0.05, n_steps),
                    method='RK45', rtol=1e-9, atol=1e-9)
    return sol.y[0].astype(np.float32)

# Define continuum
systems = [
    ('Harmonic',   simulate_harmonic,  {},                    'periodic'),
    ('VanderPol',  simulate_vanderpol, {'mu': 2.0},           'limit_cycle'),
    ('Duffing',    simulate_duffing,   {},                    'weakly_chaotic'),
    ('Rossler',    simulate_rossler,   {},                    'chaotic'),
    ('Lorenz',     simulate_lorenz,    {'rho': 28.0},         'chaotic'),
]

exp19_results = []

for name, sim_fn, kwargs, regime in systems:
    print(f'\n  Simulating {name} ({regime})...')
    series = sim_fn(n_steps=4000, **kwargs)
    series = series[500:]  # discard transient

    pe  = permutation_entropy(series, order=3, normalise=True)
    lam = rosenstein_lambda1(series, m=5, tau=1, max_iter=50)

    data_CT = series[None, :]  # (1, T)
    r = evaluate(data_CT, PRED_LEN, n_windows=N_WINDOWS,
                 label=f'{name}_{regime}')
    if r:
        r['system']  = name
        r['regime']  = regime
        r['pe']      = pe
        r['lambda1'] = lam
        exp19_results.append(r)
        print(f'    PE={pe:.3f}  lambda1={lam:.4f}  '
              f'advantage={r["advantage_mae"]:+.4f}  p={r["wilcoxon_p"]:.3f}')

df_exp19 = pd.DataFrame(exp19_results)
df_exp19.to_csv('exp19_complexity_continuum.csv', index=False)

print('\n=== Exp 19 Summary ===')
print(f'{"system":>12} | {"regime":>14} | {"PE":>5} | '
      f'{"lambda1":>8} | {"advantage":>10} | {"p":>7}')
print('-' * 70)
for _, row in df_exp19.iterrows():
    sig = '*' if row.wilcoxon_p < 0.05 else ''
    print(f'{row.system:>12} | {row.regime:>14} | {row.pe:>5.3f} | '
          f'{row.lambda1:>8.4f} | {row.advantage_mae:>10.4f} | '
          f'{row.wilcoxon_p:>6.3f}{sig}')

# Is there a threshold or continuous growth?
pes  = df_exp19.pe.values
advs = df_exp19.advantage_mae.values
rho_cont, p_cont = spearmanr(pes, advs)
print(f'\nSpearman (PE vs advantage): r={rho_cont:.3f}  p={p_cont:.3f}')

first_sig = df_exp19[df_exp19.wilcoxon_p < 0.05].iloc[0] \
    if (df_exp19.wilcoxon_p < 0.05).any() else None
if first_sig is not None:
    print(f'First significant advantage at: {first_sig.system} '
          f'({first_sig.regime})  PE={first_sig.pe:.3f}')

Exp 19: Synthetic Complexity Continuum
----------------------------------------------------------------------

  Simulating Harmonic (periodic)...


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Harmonic_periodic                                   H=  96  panda=0.0647[±0.0423]  chronos=0.4346[±0.2179]  Adv=+0.3699  p=0.004 *
    PE=0.438  lambda1=0.2393  advantage=+0.3699  p=0.004

  Simulating VanderPol (limit_cycle)...


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  VanderPol_limit_cycle                               H=  96  panda=0.0319[±0.0081]  chronos=0.0426[±0.0202]  Adv=+0.0106  p=0.027 *
    PE=0.431  lambda1=nan  advantage=+0.0106  p=0.027

  Simulating Duffing (weakly_chaotic)...


...

  Duffing_weakly_chaotic                              H=  96  panda=0.5811[±0.3238]  chronos=0.7948[±0.4953]  Adv=+0.2137  p=0.055 ~
    PE=0.476  lambda1=nan  advantage=+0.2137  p=0.055

  Simulating Rossler (chaotic)...


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Rossler_chaotic                                     H=  96  panda=0.0826[±0.0466]  chronos=0.3868[±0.2464]  Adv=+0.3042  p=0.004 *
    PE=0.442  lambda1=nan  advantage=+0.3042  p=0.004

  Simulating Lorenz (chaotic)...


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Lorenz_chaotic                                      H=  96  panda=0.0555[±0.0254]  chronos=0.5314[±0.4630]  Adv=+0.4759  p=0.004 *
    PE=0.455  lambda1=0.1434  advantage=+0.4759  p=0.004

=== Exp 19 Summary ===
      system |         regime |    PE |  lambda1 |  advantage |       p
----------------------------------------------------------------------
    Harmonic |       periodic | 0.438 |   0.2393 |     0.3699 |  0.004*
   VanderPol |    limit_cycle | 0.431 |      nan |     0.0106 |  0.027*
     Duffing | weakly_chaotic | 0.476 |      nan |     0.2137 |  0.055
     Rossler |        chaotic | 0.442 |      nan |     0.3042 |  0.004*
      Lorenz |        chaotic | 0.455 |   0.1434 |     0.4759 |  0.004*

Spearman (PE vs advantage): r=0.300  p=0.624
First significant advantage at: Harmonic (periodic)  PE=0.438


In [14]:
# Quick diagnostic: per-channel difficulty within each subset
print('Per-channel difficulty breakdown within subsets:')
for name, ch_idx in [('homo_matched', homo_matched),
                      ('mixed_matched', mixed_matched),
                      ('heterogeneous', hetero_channels)]:
    ch_diff = difficulty[ch_idx]
    print(f'  {name:>15}: mean={ch_diff.mean():.4f}  '
          f'std={ch_diff.std():.4f}  '
          f'min={ch_diff.min():.4f}  max={ch_diff.max():.4f}  '
          f'CV={ch_diff.std()/ch_diff.mean():.3f}')

Per-channel difficulty breakdown within subsets:


NameError: name 'homo_matched' is not defined

In [15]:
difficulty = np.array([
    0.4135, 0.8400, 0.8061, 0.9733, 0.9479,  # channels 0-4
    0.7382, 0.6382, 0.8595, 0.7421, 0.5061,  # channels 5-9
    0.6351, 1.0969, 0.8513, 0.7944, 0.6731,  # channels 10-14
    0.2682, 0.8337, 0.7318, 0.6885, 0.6440,  # channels 15-19
    0.9268                                     # channel 20
])

homo_matched    = np.array([6, 8, 5, 3, 7, 2, 1])
mixed_matched   = np.array([1, 3, 8, 4, 16, 2, 17])
hetero_channels = np.array([0, 4, 11, 13, 1, 2, 3])

print('Per-channel difficulty breakdown within subsets:')
for name, ch_idx in [('homo_matched',   homo_matched),
                      ('mixed_matched',  mixed_matched),
                      ('heterogeneous',  hetero_channels)]:
    ch_diff = difficulty[ch_idx]
    cv      = ch_diff.std() / ch_diff.mean()
    print(f'  {name:>15}: mean={ch_diff.mean():.4f}  std={ch_diff.std():.4f}  '
          f'min={ch_diff.min():.4f}  max={ch_diff.max():.4f}  CV={cv:.3f}')

Per-channel difficulty breakdown within subsets:
     homo_matched: mean=0.7996  std=0.0990  min=0.6382  max=0.9733  CV=0.124
    mixed_matched: mean=0.8393  std=0.0861  min=0.7318  max=0.9733  CV=0.103
    heterogeneous: mean=0.8389  std=0.2002  min=0.4135  max=1.0969  CV=0.239


In [18]:
# Build variance-controlled heterogeneous subset
# Goal: same mean difficulty (~0.84), same CV as homo/mixed (~0.10-0.12),
#       but drawn from multiple clusters (heterogeneous)

# Find 7-channel subset that:
# 1. Draws from at least 3 different clusters
# 2. Mean difficulty close to 0.8389
# 3. CV close to 0.11 (matching homo/mixed)

from itertools import combinations

best_subset  = None
best_score   = np.inf
target_mean  = 0.8389
target_cv    = 0.11
all_channels = np.arange(len(difficulty))

# Random search — exhaustive is 21C7 = 116280, manageable
rng = np.random.default_rng(SEED)
for _ in range(50000):
    sub = rng.choice(all_channels, size=7, replace=False)
    ch_diff = difficulty[sub]
    mean_d  = ch_diff.mean()
    cv_d    = ch_diff.std() / mean_d

    # Check cluster diversity — must span at least 3 clusters
    clusters_present = set(labels[sub])
    if len(clusters_present) < 3:
        continue

    # Score: penalise deviation from target mean and target CV
    score = abs(mean_d - target_mean) + abs(cv_d - target_cv)
    if score < best_score:
        best_score  = score
        best_subset = sub.copy()

hetero_controlled = best_subset
hc_diff  = difficulty[hetero_controlled]
hc_het   = intra_cluster_heterogeneity(hetero_controlled, dist_matrix)
hc_cv    = hc_diff.std() / hc_diff.mean()
clusters_in_subset = set(labels[hetero_controlled])

print('Variance-controlled heterogeneous subset:')
print(f'  Channels:    {hetero_controlled.tolist()}')
print(f'  Clusters:    {sorted(clusters_in_subset)}  (n={len(clusters_in_subset)})')
print(f'  Mean diff:   {hc_diff.mean():.4f}  (target={target_mean:.4f})')
print(f'  CV diff:     {hc_cv:.3f}    (target={target_cv:.3f})')
print(f'  Het score:   {hc_het:.4f}')
print(f'  Search score:{best_score:.4f}')

print('\nComparison:')
print(f'  {"subset":>20} | {"mean_diff":>9} | {"CV_diff":>8} | {"het":>6} | {"clusters"}')
print('-' * 65)
for name, ch_idx in [('homo_matched',    homo_matched),
                      ('mixed_matched',   mixed_matched),
                      ('hetero_original', hetero_channels),
                      ('hetero_controlled', hetero_controlled)]:
    d    = difficulty[ch_idx]
    cv   = d.std() / d.mean()
    het  = intra_cluster_heterogeneity(ch_idx, dist_matrix)
    cl   = sorted(set(labels[ch_idx]))
    print(f'  {name:>20} | {d.mean():>9.4f} | {cv:>8.3f} | {het:>6.4f} | {cl}')

# Now run evaluation on controlled subset
print('\nRunning forecast comparison on variance-controlled heterogeneous subset...')
data_hc = data_weather[hetero_controlled, :]
hc_results = []
for h in [96, 336]:
    r = evaluate(data_hc, h, n_windows=N_WINDOWS,
                 label=f'hetero_controlled_H{h}')
    if r:
        r['subset']        = 'hetero_controlled'
        r['heterogeneity'] = hc_het
        r['diff_cv']       = hc_cv
        hc_results.append(r)

df_hc = pd.DataFrame(hc_results)

print('\n=== Controlled Heterogeneous vs Difficulty-Matched Summary ===')
print(f'{"subset":>20} | {"H":>5} | {"het":>6} | {"diff_cv":>8} | '
      f'{"panda_mae":>10} | {"adv":>8} | {"p":>7}')
print('-' * 80)
# Print existing subsets from df_dm for reference
for _, row in df_dm.sort_values(['horizon','heterogeneity']).iterrows():
    print(f'  {row.subset:>18} | {int(row.horizon):>5} | '
          f'{row.heterogeneity:>6.4f} | {"---":>8} | '
          f'{row.panda_mae:>10.4f} | {row.advantage_mae:>8.4f} | '
          f'{row.wilcoxon_p:>6.4f}')
# Print new controlled result
for _, row in df_hc.iterrows():
    sig = '*' if row.wilcoxon_p < 0.05 else ''
    print(f'  {"hetero_controlled":>18} | {int(row.horizon):>5} | '
          f'{row.heterogeneity:>6.4f} | {row.diff_cv:>8.3f} | '
          f'{row.panda_mae:>10.4f} | {row.advantage_mae:>8.4f} | '
          f'{row.wilcoxon_p:>6.4f}{sig}')

df_hc.to_csv('hetero_controlled_results.csv', index=False)
print('\nSaved hetero_controlled_results.csv')

print('\n=== Key Question ===')
for h in [96, 336]:
    mixed_adv = float(df_dm[(df_dm.subset=='mixed_matched') &
                             (df_dm.horizon==h)].advantage_mae)
    hc_adv    = float(df_hc[df_hc.horizon==h].advantage_mae) \
                if len(df_hc[df_hc.horizon==h]) else np.nan
    print(f'  H={h}: mixed_adv={mixed_adv:.4f}  hetero_controlled_adv={hc_adv:.4f}')
    if not np.isnan(hc_adv):
        if hc_adv < mixed_adv - 0.05:
            obs = 'Advantage drops on heterogeneous even after CV control. H1 supported.'
        elif abs(hc_adv - mixed_adv) < 0.05:
            obs = 'Advantage stable. Original drop was a within-subset variance confound.'
        else:
            obs = 'Ambiguous.'
        print(f'  Observation: {obs}')

Variance-controlled heterogeneous subset:
  Channels:    [16, 4, 19, 7, 12, 2, 20]
  Clusters:    [1, 2, 3]  (n=3)
  Mean diff:   0.8385  (target=0.8389)
  CV diff:     0.110    (target=0.110)
  Het score:   0.8529
  Search score:0.0007

Comparison:
                subset | mean_diff |  CV_diff |    het | clusters
-----------------------------------------------------------------
          homo_matched |    0.7996 |    0.124 | 0.0857 | [1]
         mixed_matched |    0.8393 |    0.103 | 0.6021 | [1, 2]
       hetero_original |    0.8389 |    0.239 | 0.9471 | [1, 2, 3, 4]
     hetero_controlled |    0.8385 |    0.110 | 0.8529 | [1, 2, 3]

Running forecast comparison on variance-controlled heterogeneous subset...


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  hetero_controlled_H96                               H=  96  panda=0.6051[±0.2640]  chronos=0.8660[±0.2050]  Adv=+0.2609  p=0.004 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  hetero_controlled_H336                              H= 336  panda=0.8667[±0.1727]  chronos=1.1754[±0.3909]  Adv=+0.3088  p=0.012 *

=== Controlled Heterogeneous vs Difficulty-Matched Summary ===
              subset |     H |    het |  diff_cv |  panda_mae |      adv |       p
--------------------------------------------------------------------------------


NameError: name 'df_dm' is not defined

In [20]:
# Reconstruct df_dm from saved CSV — no reruns needed
df_dm = pd.read_csv('difficulty_matched_results.csv')

# Now print the summary that failed
print('=== Controlled Heterogeneous vs Difficulty-Matched Summary ===')
print(f'{"subset":>20} | {"H":>5} | {"het":>6} | {"diff_cv":>8} | '
      f'{"panda_mae":>10} | {"adv":>8} | {"p":>7}')
print('-' * 80)

for _, row in df_dm.sort_values(['horizon','heterogeneity']).iterrows():
    print(f'  {row.subset:>18} | {int(row.horizon):>5} | '
          f'{row.heterogeneity:>6.4f} | {"---":>8} | '
          f'{row.panda_mae:>10.4f} | {row.advantage_mae:>8.4f} | '
          f'{row.wilcoxon_p:>6.4f}')

for _, row in df_hc.iterrows():
    sig = '*' if row.wilcoxon_p < 0.05 else ''
    print(f'  {"hetero_controlled":>18} | {int(row.horizon):>5} | '
          f'{row.heterogeneity:>6.4f} | {row.diff_cv:>8.3f} | '
          f'{row.panda_mae:>10.4f} | {row.advantage_mae:>8.4f} | '
          f'{row.wilcoxon_p:>6.4f}{sig}')

df_hc.to_csv('hetero_controlled_results.csv', index=False)
print('\nSaved hetero_controlled_results.csv')

print('\n=== Key Question ===')
for h in [96, 336]:
    mixed_adv = float(df_dm[(df_dm.subset=='mixed_matched') &
                             (df_dm.horizon==h)].advantage_mae)
    hc_row    = df_hc[df_hc.horizon==h]
    hc_adv    = float(hc_row.advantage_mae) if len(hc_row) else np.nan
    hc_panda  = float(hc_row.panda_mae) if len(hc_row) else np.nan
    mixed_panda = float(df_dm[(df_dm.subset=='mixed_matched') &
                               (df_dm.horizon==h)].panda_mae)

    print(f'\n  H={h}:')
    print(f'    mixed_matched:     panda={mixed_panda:.4f}  adv={mixed_adv:.4f}')
    print(f'    hetero_controlled: panda={hc_panda:.4f}  adv={hc_adv:.4f}')

    if not np.isnan(hc_adv):
        panda_delta = hc_panda - mixed_panda
        adv_delta   = hc_adv - mixed_adv
        print(f'    Panda MAE delta: {panda_delta:+.4f}  Adv delta: {adv_delta:+.4f}')
        if hc_adv < mixed_adv - 0.05:
            obs = ('Advantage drops on hetero_controlled despite matched mean AND CV. '
                   'H1 supported cleanly. Run Exp 22.')
        elif abs(hc_adv - mixed_adv) < 0.05:
            obs = ('Advantage stable. Original drop was within-subset variance confound. '
                   'H1 not supported. Reconsider Exp 22.')
        else:
            obs = 'Ambiguous. Check per-horizon pattern.'
        print(f'    Observation: {obs}')

=== Controlled Heterogeneous vs Difficulty-Matched Summary ===
              subset |     H |    het |  diff_cv |  panda_mae |      adv |       p
--------------------------------------------------------------------------------
        homo_matched |    96 | 0.0857 |      --- |     0.3306 |   0.3691 | 0.0039
       mixed_matched |    96 | 0.6021 |      --- |     0.4749 |   0.3334 | 0.0039
       heterogeneous |    96 | 0.9471 |      --- |     0.6184 |   0.2178 | 0.0742
        homo_matched |   336 | 0.0857 |      --- |     0.8413 |   0.3054 | 0.0039
       mixed_matched |   336 | 0.6021 |      --- |     0.8562 |   0.0903 | 0.0195
       heterogeneous |   336 | 0.9471 |      --- |     1.1230 |   0.1199 | 0.1250
   hetero_controlled |    96 | 0.8529 |    0.110 |     0.6051 |   0.2609 | 0.0039*
   hetero_controlled |   336 | 0.8529 |    0.110 |     0.8667 |   0.3088 | 0.0117*

Saved hetero_controlled_results.csv

=== Key Question ===

  H=96:
    mixed_matched:     panda=0.4749  adv=0.3334

In [21]:
print('Chronos MAE breakdown:')
for _, row in df_hc.iterrows():
    print(f'  H={int(row.horizon)}: chronos_mae={row.chronos_mae:.4f}')

# Compare with mixed_matched Chronos
for h in [96, 336]:
    mixed_chronos = float(df_dm[(df_dm.subset=='mixed_matched') &
                                 (df_dm.horizon==h)].chronos_mae)
    hc_chronos    = float(df_hc[df_hc.horizon==h].chronos_mae) \
                    if 'chronos_mae' in df_hc.columns else np.nan
    print(f'  H={h}: mixed_chronos={mixed_chronos:.4f}  hc_chronos={hc_chronos:.4f}')

Chronos MAE breakdown:
  H=96: chronos_mae=0.8660
  H=336: chronos_mae=1.1754
  H=96: mixed_chronos=0.8083  hc_chronos=0.8660
  H=336: mixed_chronos=0.9465  hc_chronos=1.1754


In [23]:
# EXP 22: Node embedding ablation (conditional on Chronos heterogeneity result)
# Adds a learned per-channel bias offset to Panda input before patch embedding.
# Freezes all Panda weights. Only optimises identity offsets.
# Tests whether sensor identity recovers performance on heterogeneous channels.

print('Exp 22: Node Embedding Ablation')
print('CONDITIONAL: only meaningful if Panda degrades >> Chronos in heterogeneity cell')
print('-' * 70)

print('Using hetero_controlled subset (matched difficulty mean AND CV)')
print('-' * 70)

# Verify hetero_controlled is in memory
# If not, reconstruct:
if 'hetero_controlled' not in dir():
    hetero_controlled = np.array([16, 4, 19, 7, 12, 2, 20])
    homo_matched      = np.array([6, 8, 5, 3, 7, 2, 1])
    print('Reconstructed channel indices from saved values.')

def train_node_embeddings(data_CT, horizon, n_epochs=30,
                           lr=0.01, n_windows=N_WINDOWS, seed=SEED):
    C, T      = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    starts    = np.linspace(0, max_start, n_windows, dtype=int)

    offsets = torch.zeros(C, requires_grad=True)
    opt     = torch.optim.Adam([offsets], lr=lr)

    losses = []
    for epoch in range(n_epochs):
        epoch_loss = 0.0
        for s in starts:
            ctx_raw           = data_CT[:, s : s + CONTEXT_LEN]
            tgt_raw           = data_CT[:, s + CONTEXT_LEN
                                           : s + CONTEXT_LEN + horizon]
            ctx_norm, mu, std = instance_norm_window(ctx_raw)
            tgt_norm_np       = (tgt_raw - mu) / std

            ctx_t   = torch.tensor(ctx_norm, dtype=torch.float32)
            ctx_off = ctx_t + offsets.unsqueeze(1)

            pred_t = panda_model.predict(
                    ctx_off.T, horizon,
                    limit_prediction_length=False,
                    sliding_context=True,
                )
            pred_np = pred_t.squeeze().cpu().numpy()
            if pred_np.ndim == 1:
                pred_np = pred_np[None, :]
            if pred_np.shape[0] != C:
                pred_np = pred_np.T

            # Slice to exactly horizon steps — fixes the size mismatch
            pred_np = pred_np[:, :horizon]

            tgt_t   = torch.tensor(tgt_norm_np, dtype=torch.float32)
            pred_t2 = torch.tensor(pred_np,     dtype=torch.float32)

            assert pred_t2.shape == tgt_t.shape, \
                f'Shape mismatch: pred={pred_t2.shape} tgt={tgt_t.shape}'

            loss  = torch.mean((pred_t2 - tgt_t)**2)
            reg   = 0.01 * (offsets**2).sum()
            (loss + reg).backward()
            opt.step()
            opt.zero_grad()
            epoch_loss += float(loss.item())

        losses.append(epoch_loss / n_windows)
        if epoch % 10 == 0:
            print(f'  Epoch {epoch:3d}: loss={losses[-1]:.4f}  '
                  f'offset_norm={float(offsets.norm()):.4f}')

    return offsets.detach().numpy(), losses

def panda_forecast_with_offsets(context_np, horizon, offsets_np):
    ctx_off = context_np + offsets_np[:, None]
    return panda_forecast(ctx_off, horizon)

# Train offsets on full Weather (all 21 channels)
print('\nTraining node embeddings on full Weather (all 21 channels)...')
offsets_weather, train_losses = train_node_embeddings(
    data_weather, PRED_LEN, n_epochs=30, lr=0.01
)
print(f'Training loss: {train_losses[0]:.4f} → {train_losses[-1]:.4f}')
print(f'Learned offsets: mean={offsets_weather.mean():.4f}  '
      f'std={offsets_weather.std():.4f}  '
      f'range=[{offsets_weather.min():.4f}, {offsets_weather.max():.4f}]')

# Evaluate on homo_matched and hetero_controlled
# with and without offsets
print('\nEvaluating with and without learned offsets...')
exp22_results = []

subsets_22 = [
    ('homo_matched',      homo_matched,      0.0857),
    ('hetero_controlled', hetero_controlled, 0.8529),
]

for subset_name, ch_idx, het_val in subsets_22:
    data_sub    = data_weather[ch_idx, :]
    offsets_sub = offsets_weather[ch_idx]  # subset of the 21-channel offsets

    print(f'\n  Subset: {subset_name}  channels={ch_idx.tolist()}')
    for h in [96, 336]:
        # Baseline: no offsets
        r_base = evaluate(data_sub, h, n_windows=N_WINDOWS,
                          label=f'Base_{subset_name}_H{h}')

        # With offsets
        fn_off = lambda ctx, hor, off=offsets_sub: \
            panda_forecast_with_offsets(ctx, hor, off)
        r_off  = evaluate(data_sub, h, n_windows=N_WINDOWS,
                          label=f'Offset_{subset_name}_H{h}',
                          fn_a=fn_off)

        if r_base and r_off:
            delta_panda = r_base['panda_mae'] - r_off['panda_mae']
            delta_adv   = r_off['advantage_mae'] - r_base['advantage_mae']
            exp22_results.append({
                'subset':          subset_name,
                'horizon':         h,
                'heterogeneity':   het_val,
                'panda_base':      r_base['panda_mae'],
                'panda_off':       r_off['panda_mae'],
                'chronos_mae':     r_base['chronos_mae'],
                'adv_base':        r_base['advantage_mae'],
                'adv_off':         r_off['advantage_mae'],
                'delta_panda':     delta_panda,
                'delta_adv':       delta_adv,
                'p_base':          r_base['wilcoxon_p'],
                'p_off':           r_off['wilcoxon_p'],
            })

df_exp22 = pd.DataFrame(exp22_results)
df_exp22.to_csv('exp22_node_embeddings.csv', index=False)
print('\nSaved exp22_node_embeddings.csv')

print('\n=== Exp 22 Summary ===')
print(f'{"subset":>20} | {"H":>5} | {"panda_base":>10} | {"panda_off":>10} | '
      f'{"adv_base":>9} | {"adv_off":>8} | {"delta_p":>8} | {"delta_adv":>10}')
print('-' * 95)
for _, row in df_exp22.sort_values(['horizon','heterogeneity']).iterrows():
    print(f'  {row.subset:>18} | {int(row.horizon):>5} | '
          f'{row.panda_base:>10.4f} | {row.panda_off:>10.4f} | '
          f'{row.adv_base:>9.4f} | {row.adv_off:>8.4f} | '
          f'{row.delta_panda:>+8.4f} | {row.delta_adv:>+10.4f}')

print('\n=== Key Diagnostic ===')
for h in [96, 336]:
    homo_dp  = float(df_exp22[(df_exp22.subset=='homo_matched') &
                               (df_exp22.horizon==h)].delta_panda)
    hetero_dp = float(df_exp22[(df_exp22.subset=='hetero_controlled') &
                                (df_exp22.horizon==h)].delta_panda)
    print(f'\n  H={h}:')
    print(f'    homo_matched delta_panda:      {homo_dp:+.4f}')
    print(f'    hetero_controlled delta_panda: {hetero_dp:+.4f}')

    if hetero_dp > 0.02 and hetero_dp > homo_dp + 0.01:
        obs = ('Offsets help Panda MORE on heterogeneous channels than homogeneous. '
               'Node embedding direction directly motivated. '
               'G-SWaN architectural change is justified.')
    elif homo_dp > 0.02 and hetero_dp < 0.01:
        obs = ('Offsets help on homogeneous but not heterogeneous. '
               'Simple identity offsets insufficient for heterogeneous case. '
               'Richer graph structure (full G-SWaN) may be needed.')
    elif abs(hetero_dp) < 0.01 and abs(homo_dp) < 0.01:
        obs = ('Offsets do not help either subset. '
               'Node embedding approach in this form is ineffective. '
               'Reconsider architectural direction.')
    else:
        obs = 'Mixed result. Check per-horizon breakdown carefully.'
    print(f'    Observation: {obs}')

Exp 22: Node Embedding Ablation
CONDITIONAL: only meaningful if Panda degrades >> Chronos in heterogeneity cell
----------------------------------------------------------------------
Using hetero_controlled subset (matched difficulty mean AND CV)
----------------------------------------------------------------------

Training node embeddings on full Weather (all 21 channels)...
  Epoch   0: loss=2.7451  offset_norm=0.0000
  Epoch  10: loss=2.7451  offset_norm=0.0000
  Epoch  20: loss=2.7451  offset_norm=0.0000


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


Training loss: 2.7451 → 2.7451
Learned offsets: mean=0.0000  std=0.0000  range=[0.0000, 0.0000]

Evaluating with and without learned offsets...

  Subset: homo_matched  channels=[6, 8, 5, 3, 7, 2, 1]


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


KeyboardInterrupt: 

In [25]:
import inspect
print(inspect.getsource(inner_model.forward))

    def forward(
        self,
        past_values: torch.Tensor,
        past_observed_mask: torch.Tensor | None = None,
        future_values: torch.Tensor | None = None,
        output_hidden_states: bool | None = None,
        output_attentions: bool | None = None,
        channel_attention_mask: torch.Tensor | None = None,
        return_dict: bool | None = None,
    ) -> tuple | PatchTSTForPredictionOutput:
        r"""
        Parameters:
            past_values (`torch.Tensor` of shape `(bs, sequence_length, num_input_channels)`, *required*):
                Input sequence to the model
            past_observed_mask (`torch.BoolTensor` of shape `(batch_size, sequence_length, num_input_channels)`, *optional*):
                Boolean mask to indicate which `past_values` were observed and which were missing. Mask values selected
                in `[0, 1]`:

                - 1 for values that are **observed**,
                - 0 for values that are **missing** (i.e. NaNs that w

In [19]:
# EXP 23: Prediction head fine-tuning
# Freeze entire Panda encoder. Unfreeze only the linear prediction head.
# Few-shot fine-tuning on Weather. Measure delta in advantage.
# Tests: is the fixed chaotic ODE head prior a bottleneck for non-chaotic data?

print('Exp 23: Prediction Head Fine-Tuning')
print('-' * 70)

# Access underlying model through pipeline
# Try common attribute names for HuggingFace pipeline wrappers
def get_inner_model(pipeline):
    for attr in ['model', 'forecaster', 'module', '_model']:
        if hasattr(pipeline, attr):
            inner = getattr(pipeline, attr)
            if hasattr(inner, 'named_parameters'):
                return inner
    # Fallback: search __dict__
    for k, v in pipeline.__dict__.items():
        if hasattr(v, 'named_parameters'):
            print(f'  Found inner model at pipeline.{k}')
            return v
    return None

inner_model = get_inner_model(panda_model)
if inner_model is None:
    print('ERROR: Could not find inner model with named_parameters.')
    print('Available pipeline attributes:')
    for k, v in panda_model.__dict__.items():
        print(f'  {k}: {type(v)}')
    raise AttributeError('Inner model not found. Check attribute names above.')

print(f'Inner model type: {type(inner_model)}')
print(f'Total parameters: {sum(p.numel() for p in inner_model.parameters()):,}')

# Find prediction head parameters
head_params = []
head_names  = []
for name, param in inner_model.named_parameters():
    if any(k in name.lower() for k in ['head', 'projection', 'linear',
                                         'output', 'pred', 'forecast']):
        head_params.append(param)
        head_names.append(name)

if len(head_params) == 0:
    print('\nNo head params found with standard keywords.')
    print('All parameter names:')
    for name, param in inner_model.named_parameters():
        print(f'  {name}: {param.shape}')
    print('\nUsing last 20% of parameters as head proxy.')
    all_params  = list(inner_model.named_parameters())
    n_head      = max(1, len(all_params) // 5)
    head_params = [p for _, p in all_params[-n_head:]]
    head_names  = [n for n, _ in all_params[-n_head:]]

print(f'\nHead parameters ({len(head_params)} tensors):')
total_head = 0
for n, p in zip(head_names, head_params):
    total_head += p.numel()
    print(f'  {n}: {p.shape}  ({p.numel():,} params)')
print(f'Total head params: {total_head:,}')

# Save original weights
original_head = [p.data.clone() for p in head_params]

def restore_head():
    for param, orig in zip(head_params, original_head):
        param.data.copy_(orig)
    print('Original head restored.')

# --- Baseline ---
print('\nBaseline (no fine-tuning):')
base_results = {}
for h in [96, 336]:
    r = evaluate(data_weather, h, n_windows=N_WINDOWS,
                 label=f'Weather_baseline_H{h}')
    if r:
        base_results[h] = r

# --- Fine-tune head ---
T_total  = data_weather.shape[1]
T_train  = int(T_total * 0.7)
data_train = data_weather[:, :T_train]

print(f'\nFine-tuning head on first {T_train} timesteps...')

# Freeze all, unfreeze head
for param in inner_model.parameters():
    param.requires_grad_(False)
for param in head_params:
    param.requires_grad_(True)

opt    = torch.optim.Adam(head_params, lr=1e-4)
losses = []
starts = np.linspace(0, T_train - CONTEXT_LEN - PRED_LEN, N_WINDOWS, dtype=int)

inner_model.train()
for step in range(50):
    s                 = starts[step % len(starts)]
    ctx_raw           = data_train[:, s : s + CONTEXT_LEN]
    tgt_raw           = data_train[:, s + CONTEXT_LEN : s + CONTEXT_LEN + PRED_LEN]
    ctx_norm, mu, std = instance_norm_window(ctx_raw)
    tgt_norm          = torch.tensor((tgt_raw - mu) / std, dtype=torch.float32)
    ctx_t             = torch.tensor(ctx_norm.T, dtype=torch.float32)

    pred = panda_model.predict(
        ctx_t, PRED_LEN,
        limit_prediction_length=False,
        sliding_context=True,
    )
    pred = pred.squeeze()
    if pred.ndim == 1:
        pred = pred.unsqueeze(0)
    if pred.shape[0] != data_weather.shape[0]:
        pred = pred.T

    loss = torch.mean((pred - tgt_norm) ** 2)
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(float(loss.item()))
    if step % 10 == 0:
        print(f'  Step {step:3d}: loss={losses[-1]:.4f}')

inner_model.eval()
for param in inner_model.parameters():
    param.requires_grad_(True)

print(f'\nLoss: {losses[0]:.4f} → {losses[-1]:.4f}  '
      f'({100*(losses[0]-losses[-1])/losses[0]:.1f}% reduction)')

# --- Post fine-tuning evaluation ---
print('\nPost fine-tuning:')
ft_results = {}
for h in [96, 336]:
    r = evaluate(data_weather, h, n_windows=N_WINDOWS,
                 label=f'Weather_finetuned_H{h}')
    if r:
        ft_results[h] = r

restore_head()

# --- Summary ---
print('\n=== Exp 23 Summary ===')
print(f'{"H":>5} | {"panda_base":>10} | {"panda_ft":>9} | '
      f'{"adv_base":>9} | {"adv_ft":>8} | {"delta_adv":>10} | Interpretation')
print('-' * 80)
exp23_records = []
for h in [96, 336]:
    if h not in base_results or h not in ft_results:
        continue
    b    = base_results[h]
    f    = ft_results[h]
    d_p  = b['panda_mae'] - f['panda_mae']
    d_a  = f['advantage_mae'] - b['advantage_mae']
    if d_p > 0.02 and d_a > 0.02:
        interp = 'Head fine-tuning helps. Fixed head is a bottleneck.'
    elif abs(d_p) < 0.01:
        interp = 'No effect. Head prior is not the bottleneck.'
    else:
        interp = 'Marginal. Head is a partial bottleneck.'
    print(f'{h:>5} | {b["panda_mae"]:>10.4f} | {f["panda_mae"]:>9.4f} | '
          f'{b["advantage_mae"]:>9.4f} | {f["advantage_mae"]:>8.4f} | '
          f'{d_a:>+10.4f} | {interp}')
    exp23_records.append({
        'horizon': h, 'panda_base': b['panda_mae'],
        'panda_ft': f['panda_mae'], 'adv_base': b['advantage_mae'],
        'adv_ft': f['advantage_mae'], 'delta_panda': d_p, 'delta_adv': d_a,
    })

pd.DataFrame(exp23_records).to_csv('exp23_head_finetuning.csv', index=False)
print('Saved exp23_head_finetuning.csv')

Exp 23: Prediction Head Fine-Tuning
----------------------------------------------------------------------
Inner model type: <class 'panda.patchtst.patchtst.PatchTSTForPrediction'>
Total parameters: 21,354,624

Head parameters (2 tensors):
  model.encoder.embedder.projection.weight: torch.Size([512, 512])  (262,144 params)
  head.projection.weight: torch.Size([128, 512])  (65,536 params)
Total head params: 327,680

Baseline (no fine-tuning):


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Weather_baseline_H96                                H=  96  panda=0.6089[±0.2477]  chronos=0.7632[±0.0978]  Adv=+0.1543  p=0.012 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Weather_baseline_H336                               H= 336  panda=0.8697[±0.2742]  chronos=1.0590[±0.2383]  Adv=+0.1894  p=0.020 *

Fine-tuning head on first 36887 timesteps...


RuntimeError: The size of tensor a (128) must match the size of tensor b (96) at non-singleton dimension 1

In [24]:
# Continue Exp 23 from after the failed fine-tuning step
# Reuses: inner_model, head_params, original_head, base_results, opt
# Just rerun the fine-tuning loop with the slice fix

print('Fine-tuning head (continued with slice fix)...')

# Restore original head weights first in case partial updates occurred
restore_head()

# Rebuild optimizer
opt    = torch.optim.Adam(head_params, lr=1e-4)
losses = []
starts = np.linspace(0, T_train - CONTEXT_LEN - PRED_LEN, N_WINDOWS, dtype=int)

inner_model.train()
for param in inner_model.parameters():
    param.requires_grad_(False)
for param in head_params:
    param.requires_grad_(True)

for step in range(50):
    s                 = starts[step % len(starts)]
    ctx_raw           = data_train[:, s : s + CONTEXT_LEN]
    tgt_raw           = data_train[:, s + CONTEXT_LEN : s + CONTEXT_LEN + PRED_LEN]
    ctx_norm, mu, std = instance_norm_window(ctx_raw)
    tgt_norm          = torch.tensor((tgt_raw - mu) / std, dtype=torch.float32)
    ctx_t             = torch.tensor(ctx_norm.T, dtype=torch.float32)

    pred = panda_model.predict(
        ctx_t, PRED_LEN,
        limit_prediction_length=False,
        sliding_context=True,
    )
    pred = pred.squeeze()
    if pred.ndim == 1:
        pred = pred.unsqueeze(0)
    if pred.shape[0] != data_weather.shape[0]:
        pred = pred.T

    # Fix: slice to exactly PRED_LEN
    pred = pred[:, :PRED_LEN]

    loss = torch.mean((pred - tgt_norm) ** 2)
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(float(loss.item()))
    if step % 10 == 0:
        print(f'  Step {step:3d}: loss={losses[-1]:.4f}')

inner_model.eval()
for param in inner_model.parameters():
    param.requires_grad_(True)

print(f'\nLoss: {losses[0]:.4f} → {losses[-1]:.4f}  '
      f'({100*(losses[0]-losses[-1])/losses[0]:.1f}% reduction)')

# Post fine-tuning evaluation
print('\nPost fine-tuning:')
ft_results = {}
for h in [96, 336]:
    r = evaluate(data_weather, h, n_windows=N_WINDOWS,
                 label=f'Weather_finetuned_H{h}')
    if r:
        ft_results[h] = r

restore_head()

# Summary
print('\n=== Exp 23 Summary ===')
print(f'{"H":>5} | {"panda_base":>10} | {"panda_ft":>9} | '
      f'{"adv_base":>9} | {"adv_ft":>8} | {"delta_adv":>10} | Interpretation')
print('-' * 80)
exp23_records = []
for h in [96, 336]:
    if h not in base_results or h not in ft_results:
        continue
    b   = base_results[h]
    f   = ft_results[h]
    d_p = b['panda_mae'] - f['panda_mae']
    d_a = f['advantage_mae'] - b['advantage_mae']
    if d_p > 0.02 and d_a > 0.02:
        interp = 'Head fine-tuning helps. Fixed head is a bottleneck.'
    elif abs(d_p) < 0.01:
        interp = 'No effect. Head prior is not the bottleneck.'
    else:
        interp = 'Marginal. Head is a partial bottleneck.'
    print(f'{h:>5} | {b["panda_mae"]:>10.4f} | {f["panda_mae"]:>9.4f} | '
          f'{b["advantage_mae"]:>9.4f} | {f["advantage_mae"]:>8.4f} | '
          f'{d_a:>+10.4f} | {interp}')
    exp23_records.append({
        'horizon': h, 'panda_base': b['panda_mae'],
        'panda_ft': f['panda_mae'], 'adv_base': b['advantage_mae'],
        'adv_ft': f['advantage_mae'], 'delta_panda': d_p, 'delta_adv': d_a,
    })

pd.DataFrame(exp23_records).to_csv('exp23_head_finetuning.csv', index=False)
print('Saved exp23_head_finetuning.csv')

Fine-tuning head (continued with slice fix)...
Original head restored.


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [26]:
# ── EXP 22: Node Embedding Ablation ─────────────────────────────────────
print('Exp 22: Node Embedding Ablation (gradient-based)')
print('-' * 70)

def train_node_embeddings_grad(data_CT, horizon, n_epochs=30,
                                lr=0.01, n_windows=N_WINDOWS):
    C, T      = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    starts    = np.linspace(0, max_start, n_windows, dtype=int)

    offsets = torch.zeros(C, requires_grad=True)
    opt     = torch.optim.Adam([offsets], lr=lr)
    losses  = []

    inner_model.train()
    for epoch in range(n_epochs):
        epoch_loss = 0.0
        for s in starts:
            ctx_raw           = data_CT[:, s : s + CONTEXT_LEN]
            tgt_raw           = data_CT[:, s + CONTEXT_LEN
                                           : s + CONTEXT_LEN + horizon]
            ctx_norm, mu, std = instance_norm_window(ctx_raw)
            tgt_t             = torch.tensor((tgt_raw - mu) / std,
                                             dtype=torch.float32)
            ctx_t   = torch.tensor(ctx_norm, dtype=torch.float32)
            ctx_off = ctx_t + offsets.unsqueeze(1)  # (C, T)

            x    = ctx_off.T.unsqueeze(0)  # (1, T, C)
            out  = inner_model(past_values=x, return_dict=True)
            pred = out.prediction_outputs.squeeze(0).T[:, :horizon]  # (C, H)

            loss = torch.mean((pred - tgt_t) ** 2)
            opt.zero_grad()
            loss.backward()
            opt.step()
            epoch_loss += float(loss.item())

        losses.append(epoch_loss / n_windows)
        if epoch % 10 == 0:
            print(f'  Epoch {epoch:3d}: loss={losses[-1]:.4f}  '
                  f'offset_norm={float(offsets.norm()):.4f}')

    inner_model.eval()
    return offsets.detach().numpy(), losses

def panda_forecast_with_offsets(context_np, horizon, offsets_np):
    ctx_off = context_np + offsets_np[:, None]
    return panda_forecast(ctx_off, horizon)

# Train on full Weather (all 21 channels)
print('\nTraining node embeddings on full Weather...')
offsets_weather, train_losses_22 = train_node_embeddings_grad(
    data_weather, PRED_LEN, n_epochs=30, lr=0.01
)
print(f'Training loss: {train_losses_22[0]:.4f} → {train_losses_22[-1]:.4f}')
print(f'Offsets: mean={offsets_weather.mean():.4f}  '
      f'std={offsets_weather.std():.4f}  '
      f'range=[{offsets_weather.min():.4f}, {offsets_weather.max():.4f}]')

# Evaluate on homo_matched and hetero_controlled with/without offsets
exp22_results = []
for subset_name, ch_idx, het_val in [
    ('homo_matched',      homo_matched,      0.0857),
    ('hetero_controlled', hetero_controlled, 0.8529),
]:
    data_sub    = data_weather[ch_idx, :]
    offsets_sub = offsets_weather[ch_idx]
    print(f'\n  Subset: {subset_name}  channels={ch_idx.tolist()}')

    for h in [96, 336]:
        r_base = evaluate(data_sub, h, n_windows=N_WINDOWS,
                          label=f'Base_{subset_name}_H{h}')
        fn_off = lambda ctx, hor, off=offsets_sub: \
            panda_forecast_with_offsets(ctx, hor, off)
        r_off  = evaluate(data_sub, h, n_windows=N_WINDOWS,
                          label=f'Offset_{subset_name}_H{h}',
                          fn_a=fn_off)
        if r_base and r_off:
            exp22_results.append({
                'subset':      subset_name,
                'horizon':     h,
                'het':         het_val,
                'panda_base':  r_base['panda_mae'],
                'panda_off':   r_off['panda_mae'],
                'chronos_mae': r_base['chronos_mae'],
                'adv_base':    r_base['advantage_mae'],
                'adv_off':     r_off['advantage_mae'],
                'delta_panda': r_base['panda_mae'] - r_off['panda_mae'],
                'delta_adv':   r_off['advantage_mae'] - r_base['advantage_mae'],
                'p_base':      r_base['wilcoxon_p'],
                'p_off':       r_off['wilcoxon_p'],
            })

df_exp22 = pd.DataFrame(exp22_results)
df_exp22.to_csv('exp22_node_embeddings.csv', index=False)
print('\nSaved exp22_node_embeddings.csv')

print('\n=== Exp 22 Summary ===')
print(f'{"subset":>20} | {"H":>5} | {"panda_base":>10} | {"panda_off":>10} | '
      f'{"adv_base":>9} | {"adv_off":>8} | {"delta_p":>8} | {"delta_adv":>10}')
print('-' * 95)
for _, row in df_exp22.sort_values(['horizon','het']).iterrows():
    print(f'  {row.subset:>18} | {int(row.horizon):>5} | '
          f'{row.panda_base:>10.4f} | {row.panda_off:>10.4f} | '
          f'{row.adv_base:>9.4f} | {row.adv_off:>8.4f} | '
          f'{row.delta_panda:>+8.4f} | {row.delta_adv:>+10.4f}')

print('\n=== Key Diagnostic ===')
for h in [96, 336]:
    homo_dp   = float(df_exp22[(df_exp22.subset=='homo_matched') &
                                (df_exp22.horizon==h)].delta_panda)
    hetero_dp = float(df_exp22[(df_exp22.subset=='hetero_controlled') &
                                (df_exp22.horizon==h)].delta_panda)
    print(f'\n  H={h}:')
    print(f'    homo_matched delta_panda:      {homo_dp:+.4f}')
    print(f'    hetero_controlled delta_panda: {hetero_dp:+.4f}')
    if hetero_dp > 0.02 and hetero_dp > homo_dp + 0.01:
        obs = ('Offsets help MORE on heterogeneous. '
               'G-SWaN node embedding direction is directly justified.')
    elif homo_dp > 0.02 and hetero_dp < 0.01:
        obs = ('Offsets help homo but not hetero. '
               'Simple identity offsets insufficient. Full G-SWaN needed.')
    elif abs(hetero_dp) < 0.01 and abs(homo_dp) < 0.01:
        obs = ('Offsets do not help either subset. '
               'Node embedding approach ineffective in this form.')
    else:
        obs = 'Mixed result. Check per-horizon breakdown.'
    print(f'    Observation: {obs}')

Exp 22: Node Embedding Ablation (gradient-based)
----------------------------------------------------------------------

Training node embeddings on full Weather...
  Epoch   0: loss=2.7465  offset_norm=0.1097
  Epoch  10: loss=2.6889  offset_norm=0.8283
  Epoch  20: loss=2.6823  offset_norm=1.0420
Training loss: 2.7465 → 2.6808
Offsets: mean=-0.0470  std=0.2369  range=[-0.5963, 0.4331]

  Subset: homo_matched  channels=[6, 8, 5, 3, 7, 2, 1]


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Base_homo_matched_H96                               H=  96  panda=0.3276[±0.1762]  chronos=0.7137[±0.1781]  Adv=+0.3861  p=0.004 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


  Offset_homo_matched_H96                             H=  96  panda=0.4125[±0.1947]  chronos=0.8512[±0.0749]  Adv=+0.4387  p=0.004 *


...

  Offset_hetero_controlled_H96                        H=  96  panda=0.5677[±0.2488]  chronos=0.8443[±0.0928]  Adv=+0.2767  p=0.004 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Base_hetero_controlled_H336                         H= 336  panda=0.8667[±0.1727]  chronos=0.9542[±0.2408]  Adv=+0.0876  p=0.020 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Offset_hetero_controlled_H336                       H= 336  panda=0.8701[±0.1686]  chronos=1.0161[±0.2521]  Adv=+0.1460  p=0.004 *

Saved exp22_node_embeddings.csv

=== Exp 22 Summary ===
              subset |     H | panda_base |  panda_off |  adv_base |  adv_off |  delta_p |  delta_adv
-----------------------------------------------------------------------------------------------
        homo_matched |    96 |     0.3276 |     0.4125 |    0.3861 |   0.4387 |  -0.0849 |    +0.0527
   hetero_controlled |    96 |     0.6051 |     0.5677 |    0.2422 |   0.2767 |  +0.0375 |    +0.0344
        homo_matched |   336 |     0.8389 |     0.8819 |    0.3685 |   0.1436 |  -0.0430 |    -0.2248
   hetero_controlled |   336 |     0.8667 |     0.8701 |    0.0876 |   0.1460 |  -0.0035 |    +0.0584

=== Key Diagnostic ===

  H=96:
    homo_matched delta_panda:      -0.0849
    hetero_controlled delta_panda: +0.0375
    Observation: Offsets help MORE on heterogeneous. G-SWaN node embedding direction 

In [27]:
# ── EXP 23: Prediction Head Fine-Tuning ─────────────────────────────────
print('Exp 23: Prediction Head Fine-Tuning (gradient-based)')
print('-' * 70)

# Restore head to original weights before starting
restore_head()

def finetune_head_grad(data_CT, horizon, n_steps=50, lr=1e-4,
                       n_windows=N_WINDOWS):
    C, T      = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    starts    = np.linspace(0, max_start, n_windows, dtype=int)

    for param in inner_model.parameters():
        param.requires_grad_(False)
    for param in head_params:
        param.requires_grad_(True)

    opt    = torch.optim.Adam(head_params, lr=lr)
    losses = []

    inner_model.train()
    for step in range(n_steps):
        s                 = starts[step % len(starts)]
        ctx_raw           = data_CT[:, s : s + CONTEXT_LEN]
        tgt_raw           = data_CT[:, s + CONTEXT_LEN
                                       : s + CONTEXT_LEN + horizon]
        ctx_norm, mu, std = instance_norm_window(ctx_raw)
        tgt_t             = torch.tensor((tgt_raw - mu) / std,
                                         dtype=torch.float32)

        x    = torch.tensor(ctx_norm.T, dtype=torch.float32).unsqueeze(0)
        out  = inner_model(past_values=x, return_dict=True)
        pred = out.prediction_outputs.squeeze(0).T[:, :horizon]

        loss = torch.mean((pred - tgt_t) ** 2)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(float(loss.item()))
        if step % 10 == 0:
            print(f'  Step {step:3d}: loss={losses[-1]:.4f}')

    inner_model.eval()
    for param in inner_model.parameters():
        param.requires_grad_(True)

    return losses

# Use first 70% of Weather for fine-tuning
T_total    = data_weather.shape[1]
T_train    = int(T_total * 0.7)
data_train = data_weather[:, :T_train]

print(f'Fine-tuning on first {T_train} timesteps ({T_train/T_total*100:.0f}%)...')
losses_23 = finetune_head_grad(data_train, PRED_LEN, n_steps=50, lr=1e-4)
print(f'Loss: {losses_23[0]:.4f} → {losses_23[-1]:.4f}  '
      f'({100*(losses_23[0]-losses_23[-1])/losses_23[0]:.1f}% reduction)')

# Evaluate post fine-tuning
print('\nPost fine-tuning evaluation:')
ft_results = {}
for h in [96, 336]:
    r = evaluate(data_weather, h, n_windows=N_WINDOWS,
                 label=f'Weather_finetuned_H{h}')
    if r:
        ft_results[h] = r

# Restore original head
restore_head()
print('Head restored to original weights.')

# Summary
print('\n=== Exp 23 Summary ===')
print(f'{"H":>5} | {"panda_base":>10} | {"panda_ft":>9} | '
      f'{"adv_base":>9} | {"adv_ft":>8} | {"delta_adv":>10} | Interpretation')
print('-' * 82)
exp23_records = []
for h in [96, 336]:
    if h not in base_results or h not in ft_results:
        continue
    b   = base_results[h]
    f   = ft_results[h]
    d_p = b['panda_mae'] - f['panda_mae']
    d_a = f['advantage_mae'] - b['advantage_mae']
    if d_p > 0.02 and d_a > 0.02:
        interp = 'Head fine-tuning helps. Fixed head is a bottleneck.'
    elif abs(d_p) < 0.01:
        interp = 'No effect. Head prior is not the bottleneck.'
    else:
        interp = 'Marginal. Head is a partial bottleneck.'
    print(f'{h:>5} | {b["panda_mae"]:>10.4f} | {f["panda_mae"]:>9.4f} | '
          f'{b["advantage_mae"]:>9.4f} | {f["advantage_mae"]:>8.4f} | '
          f'{d_a:>+10.4f} | {interp}')
    exp23_records.append({
        'horizon':     h,
        'panda_base':  b['panda_mae'],
        'panda_ft':    f['panda_mae'],
        'adv_base':    b['advantage_mae'],
        'adv_ft':      f['advantage_mae'],
        'delta_panda': d_p,
        'delta_adv':   d_a,
    })

pd.DataFrame(exp23_records).to_csv('exp23_head_finetuning.csv', index=False)
print('Saved exp23_head_finetuning.csv')

Exp 23: Prediction Head Fine-Tuning (gradient-based)
----------------------------------------------------------------------
Original head restored.
Fine-tuning on first 36887 timesteps (70%)...
  Step   0: loss=17.0073
  Step  10: loss=1.4874
  Step  20: loss=1.4470
  Step  30: loss=1.4235
  Step  40: loss=16.1128
Loss: 17.0073 → 48859124334592.0000  (-287283292886395.4% reduction)

Post fine-tuning evaluation:


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Weather_finetuned_H96                               H=  96  panda=0.6995[±0.2564]  chronos=0.7710[±0.2088]  Adv=+0.0715  p=0.039 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Weather_finetuned_H336                              H= 336  panda=0.8814[±0.2937]  chronos=1.0797[±0.2791]  Adv=+0.1982  p=0.020 *
Original head restored.
Head restored to original weights.

=== Exp 23 Summary ===
    H | panda_base |  panda_ft |  adv_base |   adv_ft |  delta_adv | Interpretation
----------------------------------------------------------------------------------
   96 |     0.6089 |    0.6995 |    0.1543 |   0.0715 |    -0.0828 | Marginal. Head is a partial bottleneck.
  336 |     0.8697 |    0.8814 |    0.1894 |   0.1982 |    +0.0089 | Marginal. Head is a partial bottleneck.
Saved exp23_head_finetuning.csv


In [28]:
# TOPOLOGY: Correlation dimension + permutation entropy per dataset
# Tests whether Weather is topologically closer to Lorenz than ETTh is.
# Motivates or demotivates topological flow matching direction.

print('Topology Analysis: Correlation Dimension + PE per Dataset')
print('-' * 70)

def correlation_dimension_estimate(series, emb_dim=3, tau=1,
                                    n_pairs=2000, seed=SEED):
    """
    Grassberger-Procaccia correlation dimension estimate.
    Slope of log C(r) vs log r in the scaling region.
    Returns estimated dimension.
    """
    rng    = np.random.default_rng(seed)
    N      = len(series)
    n_emb  = N - (emb_dim - 1) * tau
    if n_emb < 100:
        return np.nan

    embedded = np.array([
        series[i : i + (emb_dim-1)*tau + 1 : tau]
        for i in range(n_emb)
    ])

    # Sample random pairs
    n_pairs  = min(n_pairs, n_emb*(n_emb-1)//2)
    idx1     = rng.integers(0, n_emb, n_pairs)
    idx2     = rng.integers(0, n_emb, n_pairs)
    mask     = idx1 != idx2
    idx1, idx2 = idx1[mask], idx2[mask]
    dists    = np.linalg.norm(embedded[idx1] - embedded[idx2], axis=1)

    # Correlation integral C(r) at log-spaced r values
    r_vals  = np.logspace(
        np.log10(np.percentile(dists, 5)),
        np.log10(np.percentile(dists, 95)),
        20
    )
    C_vals  = np.array([np.mean(dists < r) for r in r_vals])
    valid   = (C_vals > 0.01) & (C_vals < 0.99)

    if valid.sum() < 4:
        return np.nan

    slope, _, _, _, _ = linregress(
        np.log(r_vals[valid]), np.log(C_vals[valid])
    )
    return float(slope)

datasets = {
    'Weather' : data_weather,
    'ETTh1'   : load_ts(f'{DATA_DIR}/ETTh1.csv'),
    'ETTh2'   : load_ts(f'{DATA_DIR}/ETTh2.csv'),
}

# Add Lorenz and Burgers as reference points
lorenz_ref  = simulate_lorenz(n_steps=3000, rho=28.0)[None, :]
burgers_ref = simulate_burgers_stable(T=500, N_x=64, nu=0.05).T[:8]

datasets['Lorenz_rho28']  = lorenz_ref
datasets['Burgers_nu0.05'] = burgers_ref

topo_records = []
print(f'\n{"Dataset":>18} | {"Corr_dim":>9} | {"PE_mean":>8} | {"n_channels"}')
print('-' * 55)

for name, data_CT in datasets.items():
    # Mean PE across first 8 channels (or fewer)
    n_ch    = min(8, data_CT.shape[0])
    pe_vals = [permutation_entropy(data_CT[c, :2000], order=3)
               for c in range(n_ch)]
    pe_mean = float(np.nanmean(pe_vals))

    # Correlation dimension on mean series or first channel
    series_for_dim = data_CT[0, :2000].astype(float)
    cd = correlation_dimension_estimate(series_for_dim, emb_dim=5, tau=2)

    print(f'{name:>18} | {cd:>9.3f} | {pe_mean:>8.3f} | {data_CT.shape[0]}')
    topo_records.append({
        'dataset': name, 'corr_dim': cd,
        'pe_mean': pe_mean, 'n_channels': data_CT.shape[0]
    })

df_topo = pd.DataFrame(topo_records)
df_topo.to_csv('topology_analysis.csv', index=False)
print('\nSaved topology_analysis.csv')

# Is Weather closer to Lorenz than ETTh is?
weather_pe  = float(df_topo[df_topo.dataset=='Weather'].pe_mean)
ett1_pe     = float(df_topo[df_topo.dataset=='ETTh1'].pe_mean)
lorenz_pe   = float(df_topo[df_topo.dataset=='Lorenz_rho28'].pe_mean)

d_weather   = abs(weather_pe - lorenz_pe)
d_ett1      = abs(ett1_pe    - lorenz_pe)

print(f'\n=== Topology Summary ===')
print(f'Distance (Weather PE - Lorenz PE): {d_weather:.3f}')
print(f'Distance (ETTh1 PE  - Lorenz PE): {d_ett1:.3f}')

if d_weather < d_ett1:
    obs = ('Weather is topologically closer to Lorenz than ETTh1 is. '
           'Topology analysis supports the distribution-proximity explanation '
           'for the Weather advantage. Topological flow matching direction motivated.')
else:
    obs = ('ETTh1 is closer to Lorenz than Weather is, or similar distance. '
           'Topology does not explain the Weather-specific advantage. '
           'Topological flow matching direction is not supported by this metric.')
print(f'Observation: {obs}')

Topology Analysis: Correlation Dimension + PE per Dataset
----------------------------------------------------------------------

           Dataset |  Corr_dim |  PE_mean | n_channels
-------------------------------------------------------
           Weather |     0.888 |    0.849 | 21
             ETTh1 |     1.541 |    0.965 | 7
             ETTh2 |     1.617 |    0.821 | 7
      Lorenz_rho28 |     0.862 |    0.466 | 1
    Burgers_nu0.05 |     0.741 |    0.071 | 8

Saved topology_analysis.csv

=== Topology Summary ===
Distance (Weather PE - Lorenz PE): 0.383
Distance (ETTh1 PE  - Lorenz PE): 0.498
Observation: Weather is topologically closer to Lorenz than ETTh1 is. Topology analysis supports the distribution-proximity explanation for the Weather advantage. Topological flow matching direction motivated.


In [29]:
# Burgers non-chaotic PDE mechanism investigation
# Univariate ablation at nu=1.0 and nu=2.0
# Same design as Experiment 9 (Weather univariate ablation)

print('Burgers Non-Chaotic PDE Mechanism: Univariate Ablation')
print('Question: Is channel attention driving the non-chaotic Burgers advantage?')
print('-' * 70)

def panda_forecast_univariate_burgers(context_np, horizon):
    """Each PCA channel processed independently — suppresses cross-channel attention."""
    C    = context_np.shape[0]
    preds = []
    for c in range(C):
        ctx_c = context_np[c:c+1, :]
        ctx_t = torch.tensor(ctx_c.T, dtype=torch.float32)
        with torch.no_grad():
            pred = panda_model.predict(
                ctx_t, horizon,
                limit_prediction_length=False,
                sliding_context=True,
            )
        p = pred.squeeze().cpu().numpy()
        if p.ndim == 0:
            p = np.array([float(p)])
        preds.append(p[:horizon])
    return np.stack(preds, axis=0)

burgers_uni_results = []

for nu in [2.0, 1.0, 0.5]:
    print(f'\n  nu={nu}:')
    U        = simulate_burgers_stable(T=1500, N_x=128, nu=nu)
    pca_data = pca_reduction(U, 16)
    data_CT  = pca_data.T  # (16, T)

    # Multivariate Panda (standard)
    r_multi = evaluate(data_CT, PRED_LEN, n_windows=N_WINDOWS,
                       label=f'Burgers_nu{nu}_multi')

    # Univariate Panda (channel attention suppressed)
    r_uni = evaluate(data_CT, PRED_LEN, n_windows=N_WINDOWS,
                     label=f'Burgers_nu{nu}_uni',
                     fn_a=panda_forecast_univariate_burgers,
                     name_a='panda_uni', name_b='chronos')

    if r_multi and r_uni:
        adv_multi = r_multi['advantage_mae']
        adv_uni   = r_uni['advantage_mae']
        p_multi   = r_multi['panda_mae']
        p_uni     = r_uni['panda_mae']
        delta     = p_uni - p_multi  # positive = channel attention helps

        print(f'    Panda multi MAE: {p_multi:.4f}  adv={adv_multi:.4f}')
        print(f'    Panda uni   MAE: {p_uni:.4f}  adv={adv_uni:.4f}')
        print(f'    Delta (uni-multi): {delta:+.4f}  '
              f'(negative = channel attention helps)')

        burgers_uni_results.append({
            'nu': nu, 'panda_multi': p_multi, 'panda_uni': p_uni,
            'adv_multi': adv_multi, 'adv_uni': adv_uni, 'delta': delta,
            'p_multi': r_multi['wilcoxon_p'], 'p_uni': r_uni['wilcoxon_p'],
        })

df_bu = pd.DataFrame(burgers_uni_results)
df_bu.to_csv('burgers_univariate_ablation.csv', index=False)
print('\nSaved burgers_univariate_ablation.csv')

print('\n=== Burgers Univariate Ablation Summary ===')
print(f'{"nu":>6} | {"p_multi":>8} | {"p_uni":>8} | '
      f'{"adv_multi":>10} | {"adv_uni":>9} | {"delta":>7} | Interpretation')
print('-' * 75)
for _, row in df_bu.iterrows():
    if row.delta < -0.02:
        interp = 'Channel attention helps. Spatial coupling is the driver.'
    elif abs(row.delta) < 0.02:
        interp = 'Channel attention neutral. Temporal architecture is the driver.'
    else:
        interp = 'Channel attention hurts. Chronos weakness is the driver.'
    print(f'{row.nu:>6} | {row.panda_multi:>8.4f} | {row.panda_uni:>8.4f} | '
          f'{row.adv_multi:>10.4f} | {row.adv_uni:>9.4f} | {row.delta:>+7.4f} | {interp}')

# Cross-compare with Weather univariate result (Exp 9)
print('\nCross-reference with Weather (Exp 9):')
print('  Weather H=96:  panda_uni=0.5541  panda_multi=0.6113  delta=+0.057')
print('  Weather H=336: panda_uni=0.8467  panda_multi=0.8762  delta=+0.030')
print('  (positive delta = channel attention hurts on Weather)')
print('\n  If Burgers delta is negative (channel attention helps) but')
print('  Weather delta is positive (channel attention hurts):')
print('  → Channel attention is specifically useful for spatially coupled PDEs')
print('    but counterproductive for heterogeneous real-world sensors.')

Burgers Non-Chaotic PDE Mechanism: Univariate Ablation
Question: Is channel attention driving the non-chaotic Burgers advantage?
----------------------------------------------------------------------

  nu=2.0:


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Burgers_nu2.0_multi                                 H=  96  panda=0.0094[±0.0042]  chronos=0.0152[±0.0205]  Adv=+0.0058  p=0.055 ~


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Burgers_nu2.0_uni                                   H=  96  panda_uni=0.0078[±0.0059]  chronos=0.0088[±0.0107]  Adv=+0.0010  p=0.273


KeyError: 'panda_mae'

In [30]:
# Nu=2.0 results already observed — hardcode them
burgers_uni_results = [
    {
        'nu': 2.0,
        'panda_multi': 0.0094, 'panda_uni': 0.0078,
        'adv_multi': 0.0058,   'adv_uni': 0.0010,
        'delta': 0.0078 - 0.0094,  # = -0.0016
        'p_multi': 0.055,           'p_uni': 0.273,
    }
]

for nu in [1.0, 0.5]:
    print(f'\n  nu={nu}:')
    U        = simulate_burgers_stable(T=1500, N_x=128, nu=nu)
    pca_data = pca_reduction(U, 16)
    data_CT  = pca_data.T

    r_multi = evaluate(data_CT, PRED_LEN, n_windows=N_WINDOWS,
                       label=f'Burgers_nu{nu}_multi')

    r_uni = evaluate(data_CT, PRED_LEN, n_windows=N_WINDOWS,
                     label=f'Burgers_nu{nu}_uni',
                     fn_a=panda_forecast_univariate_burgers,
                     fn_b=chronos_forecast,
                     name_a='panda_uni', name_b='chronos')

    if r_multi and r_uni:
        adv_multi = r_multi['advantage_mae']
        adv_uni   = r_uni['advantage_mae']
        p_multi   = r_multi['panda_mae']
        p_uni     = r_uni['panda_uni_mae']
        delta     = p_uni - p_multi

        print(f'    Panda multi MAE: {p_multi:.4f}  adv={adv_multi:.4f}')
        print(f'    Panda uni   MAE: {p_uni:.4f}  adv={adv_uni:.4f}')
        print(f'    Delta (uni-multi): {delta:+.4f}')

        burgers_uni_results.append({
            'nu': nu, 'panda_multi': p_multi, 'panda_uni': p_uni,
            'adv_multi': adv_multi, 'adv_uni': adv_uni, 'delta': delta,
            'p_multi': r_multi['wilcoxon_p'], 'p_uni': r_uni['wilcoxon_p'],
        })

df_bu = pd.DataFrame(burgers_uni_results)
df_bu.to_csv('burgers_univariate_ablation.csv', index=False)
print('\nSaved burgers_univariate_ablation.csv')

print('\n=== Burgers Univariate Ablation Summary ===')
print(f'{"nu":>6} | {"p_multi":>8} | {"p_uni":>8} | '
      f'{"adv_multi":>10} | {"adv_uni":>9} | {"delta":>7}')
print('-' * 60)
for _, row in df_bu.iterrows():
    print(f'{row.nu:>6} | {row.panda_multi:>8.4f} | {row.panda_uni:>8.4f} | '
          f'{row.adv_multi:>10.4f} | {row.adv_uni:>9.4f} | {row.delta:>+7.4f}')


  nu=1.0:


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Burgers_nu1.0_multi                                 H=  96  panda=0.0165[±0.0043]  chronos=0.0147[±0.0179]  Adv=-0.0019  p=0.320


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Burgers_nu1.0_uni                                   H=  96  panda_uni=0.0126[±0.0064]  chronos=0.0160[±0.0207]  Adv=+0.0034  p=0.004 *
    Panda multi MAE: 0.0165  adv=-0.0019
    Panda uni   MAE: 0.0126  adv=0.0034
    Delta (uni-multi): -0.0040

  nu=0.5:


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Burgers_nu0.5_multi                                 H=  96  panda=0.0235[±0.0017]  chronos=0.0566[±0.0307]  Adv=+0.0331  p=0.004 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Burgers_nu0.5_uni                                   H=  96  panda_uni=0.0241[±0.0047]  chronos=0.0473[±0.1178]  Adv=+0.0231  p=0.004 *
    Panda multi MAE: 0.0235  adv=0.0331
    Panda uni   MAE: 0.0241  adv=0.0231
    Delta (uni-multi): +0.0006

Saved burgers_univariate_ablation.csv

=== Burgers Univariate Ablation Summary ===
    nu |  p_multi |    p_uni |  adv_multi |   adv_uni |   delta
------------------------------------------------------------
   2.0 |   0.0094 |   0.0078 |     0.0058 |    0.0010 | -0.0016
   1.0 |   0.0165 |   0.0126 |    -0.0019 |    0.0034 | -0.0040
   0.5 |   0.0235 |   0.0241 |     0.0331 |    0.0231 | +0.0006
